In [217]:
import math
import json
import re
from pathlib import Path
from urllib.request import urlopen
import numpy as np
import pandas as pd
import unicodedata

In [319]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [136]:
def strip_accents(s):
   return ''.join(c for c in unicodedata.normalize('NFD', s)
                  if unicodedata.category(c) != 'Mn')

def norm_ra(a):
    # (d, i) = math.modf(a)
    # if i < 0:
    #     while i < 0:
    #         i += 360
    
    # if i >= 360:
    #     while i >= 360:
    #         i -= 360

    # return i + d
    return round(a % 360, 4)

def norm_ra_dec(a):
    return [norm_ra(a[0]), a[1]]

def norm_ra_dec_scale(a):
    return [norm_ra(a[0]), a[1], a[2]]

# IAU constellations

Sources:

- D3 Celestial constellation metadata + boundary polygons
- Stellarium modern_iau skyculture lines using HIP ids

In [62]:
D3_CONSTELLATIONS_URL = "https://raw.githubusercontent.com/ofrohn/d3-celestial/master/data/constellations.json"
D3_BOUNDS_URL = "https://raw.githubusercontent.com/ofrohn/d3-celestial/master/data/constellations.bounds.json"
STELLARIUM_MODERN_IAU_URL = "https://raw.githubusercontent.com/Stellarium/stellarium/master/skycultures/modern_iau/index.json"

with urlopen(D3_CONSTELLATIONS_URL) as response:
    d3_constellations = json.load(response)

with urlopen(D3_BOUNDS_URL) as response:
    d3_bounds = json.load(response)

with urlopen(STELLARIUM_MODERN_IAU_URL) as response:
    stellarium_modern_iau = json.load(response)

len(d3_constellations["features"]), len(d3_bounds["features"]), len(stellarium_modern_iau["constellations"])

(89, 89, 88)

## Base metadata for each constellation

In [420]:
# check that id is the same as IAU designation
for b in d3_constellations["features"]:
    if b['id'] != b['properties'].get('desig'):
        print(b['id'], b['properties']['desig'])

In [421]:
const_rows = []
for feature in d3_constellations["features"]:
    props = feature["properties"]
    label_coord = feature["geometry"].get("coordinates")
    const_rows.append({
        "id": props.get("desig"),
        "name": props.get("name"),
        "gen": props.get("gen"),
        "meaning_1": props.get("en"),
        
        # normalize rank to start from 0
        "rank": int(props["rank"]) - 1 if str(props.get("rank", "")).isdigit() else props.get("rank"),

        # preset view when looking at this constellation [ra, dec, scale]
        # will probably have to fine tune this later
        # "focus": norm_ra_dec_scale(props.get("display")),
        "focus": props.get("display"),
        
        # label position, [ra, dec]
        # will probably have to fine tune this later
        # "label": norm_ra_dec(label_coord)
        "label": label_coord
    })

const = pd.DataFrame(const_rows).sort_values("id").reset_index(drop=True)
const


,id,name,gen,meaning_1,rank,focus,label
0,And,Andromeda,Andromedae,Andromeda,0,"[-350, 37, 115]","[0.75, 43]"
1,Ant,Antlia,Antliae,Air Pump,2,"[154.1067, -32.4836, 60]","[156, -36]"
2,Aps,Apus,Apodis,Bird of Paradise,2,"[-118, -75, 47]","[-120, -74]"
3,Aql,Aquila,Aquilae,Eagle,0,"[-64.9946, 3.4109, 64]","[-69, 8]"
4,Aqr,Aquarius,Aquarii,Aquarius,1,"[-25, -11, 115]","[-22.5, -5]"
5,Ara,Ara,Arae,Altar,2,"[-99.3786, -56.5883, 59]","[-102, -56]"
6,Ari,Aries,Arietis,Ram,0,"[39.5412, 20.7923, 61]","[42, 22]"
7,Aur,Auriga,Aurigae,Charioteer,0,"[91, 42, 88]","[82.5, 37]"
8,Boo,Boötes,Boötis,Herdsman,0,"[-140, 31, 108]","[-136.5, 35]"
9,CMa,Canis Major,Canis Majoris,Great Dog,0,"[102.4362, -22.1403, 54]","[97.5, -26]"


## More info and constellation lines

From stellarium

In [422]:
stellarium_rows = []
for item in stellarium_modern_iau["constellations"]:
    abbreviation = item["id"].split()[-1]
    common_name = item.get("common_name", {})
    stellarium_rows.append({
        "id": abbreviation,
        "native": common_name.get("native"),
        "meaning_2": common_name.get("english"),
        "meaning_3": common_name.get("byname"),
        # "byname": common_name.get("byname"),
        "lines_hip": item.get("lines", []),
    })

stellarium = pd.DataFrame(stellarium_rows).sort_values("id").reset_index(drop=True)
stellarium


,id,native,meaning_2,meaning_3,lines_hip
0,And,Andromeda,Andromeda,Chained Maiden,"[[677, 3092, 5447, 9640], [113726, 116631, 116..."
1,Ant,Antlia,Air Pump,None,"[[53502, 51172, 46515]]"
2,Aps,Apus,Bird of Paradise,None,"[[72370, 81065], [80047, 81852, 81065]]"
3,Aql,Aquila,Eagle,None,"[[98036, 97649, 97278, 95501, 93805, 95501, 93..."
4,Aqr,Aquarius,Water Bearer,None,"[[102618, 106278, 109074, 110395, 110960, 1114..."
5,Ara,Ara,Altar,None,"[[85267, 85727, 82363, 83081, 83153, 85792, 88..."
6,Ari,Aries,Ram,None,"[[8832, 8903, 9884, 13209]]"
7,Aur,Auriga,Charioteer,None,"[[25428, 23015, 23767, 24608, 28360, 28380, 25..."
8,Boo,Boötes,Herdsman,None,"[[69673, 72105, 74666, 73555, 71075, 71053, 69..."
9,CMa,Canis Major,Greater Dog,None,"[[30324, 32349, 34444, 33579, 34444, 35904], [..."


## Constellation boundaries

In [423]:
# check all boundary geometries are polygon type
set(map(lambda x : x['geometry'].get("type"), d3_bounds["features"]))

{'Polygon'}

In [424]:
boundary_rows = {}
ids = set()
for feature in d3_bounds["features"]:
    abbreviation = feature["id"]
    geometry = feature["geometry"]
    polygon_list = geometry["coordinates"]

    rings = []
    for ring in polygon_list:
        corner_ring = []
        for coord in ring:
            ra = float(coord[0])
            dec = float(coord[1])
            corner_ring.append([ra, dec])

        # no point repeating the end point
        if corner_ring[0] == corner_ring[-1]:
            corner_ring.pop()
        
        rings.append(corner_ring)
    
    if abbreviation not in ids:
        ids.add(abbreviation)
        
        boundary_rows[abbreviation] = {
            "id": abbreviation,
            "boundaries": rings,
        }
    else:
        # Ser has multiple "rings"
        adding_to = boundary_rows[abbreviation]

        adding_to["boundaries"].extend(rings)

boundary = pd.DataFrame(boundary_rows.values()).sort_values("id").reset_index(drop=True)
boundary


,id,boundaries
0,And,"[[[-15.5347, 35.1682], [-15.6571, 53.168], [-8..."
1,Ant,"[[[141.9043, -24.5425], [141.7716, -37.292], [..."
2,Aps,"[[[-150.8889, -83.1201], [-83.134, -82.4583], ..."
3,Aql,"[[[-79.6498, 0.1155], [-79.6738, 2.1153], [-75..."
4,Aqr,"[[[-50.4012, 0.4362], [-50.4201, 2.4361], [-45..."
5,Ara,"[[[-110.9653, -60.2645], [-111.4294, -45.7671]..."
6,Ari,"[[[31.6652, 10.5144], [26.6557, 10.5432], [26...."
7,Aur,"[[[69.4869, 30.9219], [69.5738, 36.2547], [72...."
8,Boo,"[[[-132.2185, 7.5254], [-155.9362, 7.3606], [-..."
9,CMa,"[[[93.2156, -11.0302], [111.9734, -11.2521], [..."


## Brightest star per constellation

In [155]:
HYG_URL = "https://raw.githubusercontent.com/astronexus/HYG-Database/main/hyg/CURRENT/hygdata_v41.csv"
hyg = pd.read_csv(HYG_URL)

In [425]:
hyg_con = hyg[hyg["con"].notna()].copy()
hyg_con["mag_sort"] = pd.to_numeric(hyg_con["mag"], errors="coerce")
hyg_con = hyg_con[hyg_con["mag_sort"].notna()]

brightest_rows = []
for abbreviation, group in hyg_con.groupby("con"):
    row = group.sort_values("mag_sort").iloc[0]
    brightest_rows.append({
        "id": abbreviation,
        "brightest_hip": int(row["hip"]) if pd.notna(row.get("hip")) else None,
    })

brightest = pd.DataFrame(brightest_rows).sort_values("id").reset_index(drop=True)
brightest

,id,brightest_hip
0,And,5447
1,Ant,51172
2,Aps,72370
3,Aql,97649
4,Aqr,106278
5,Ara,85792
6,Ari,9884
7,Aur,24608
8,Boo,69673
9,CMa,32349


## Merge all constellation data

In [426]:
constellations = const.merge(stellarium, on="id", how="left")
constellations = constellations.merge(boundary, on="id", how="left")
constellations = constellations.merge(brightest, on="id", how="left")


In [427]:
# check if name differs from native
constellations.loc[constellations["name"] != constellations["native"]]

,id,name,gen,meaning_1,rank,focus,label,native,meaning_2,meaning_3,lines_hip,boundaries,brightest_hip
25,CrA,Corona Austrina,Coronae Austrini,Southern Crown,2,"[-80.3024, -41.1475, 55]","[-78, -40]",Corona Australis,Southern Crown,None,"[[93825, 94114, 94160, 94005, 90982]]","[[[-90.3745, -37.0175], [-70.4037, -36.7786], ...",94160
73,Ser,Serpens Cauda,Serpentis,Serpent,2,"[-104, 5, 145]","[-79.5, 3]",Serpens,Serpent,None,"[[77233, 78072, 77450, 76852, 77233, 76276, 77...","[[[-132.147, -0.4743], [-132.2185, 7.5254], [-...",77070
74,Ser,Serpens Caput,Serpentis,Serpent,2,"[-104, 5, 145]","[-127.5, 5]",Serpens,Serpent,None,"[[77233, 78072, 77450, 76852, 77233, 76276, 77...","[[[-132.147, -0.4743], [-132.2185, 7.5254], [-...",77070


In [428]:
constellations.at[25, 'name'] = "Corona Australis"
constellations.at[25, 'gen'] = "Coronae Australis"

# caput is bigger
constellations.at[74, 'name'] = "Serpens"
constellations = constellations.drop(73)
constellations

,id,name,gen,meaning_1,rank,focus,label,native,meaning_2,meaning_3,lines_hip,boundaries,brightest_hip
0,And,Andromeda,Andromedae,Andromeda,0,"[-350, 37, 115]","[0.75, 43]",Andromeda,Andromeda,Chained Maiden,"[[677, 3092, 5447, 9640], [113726, 116631, 116...","[[[-15.5347, 35.1682], [-15.6571, 53.168], [-8...",5447
1,Ant,Antlia,Antliae,Air Pump,2,"[154.1067, -32.4836, 60]","[156, -36]",Antlia,Air Pump,None,"[[53502, 51172, 46515]]","[[[141.9043, -24.5425], [141.7716, -37.292], [...",51172
2,Aps,Apus,Apodis,Bird of Paradise,2,"[-118, -75, 47]","[-120, -74]",Apus,Bird of Paradise,None,"[[72370, 81065], [80047, 81852, 81065]]","[[[-150.8889, -83.1201], [-83.134, -82.4583], ...",72370
3,Aql,Aquila,Aquilae,Eagle,0,"[-64.9946, 3.4109, 64]","[-69, 8]",Aquila,Eagle,None,"[[98036, 97649, 97278, 95501, 93805, 95501, 93...","[[[-79.6498, 0.1155], [-79.6738, 2.1153], [-75...",97649
4,Aqr,Aquarius,Aquarii,Aquarius,1,"[-25, -11, 115]","[-22.5, -5]",Aquarius,Water Bearer,None,"[[102618, 106278, 109074, 110395, 110960, 1114...","[[[-50.4012, 0.4362], [-50.4201, 2.4361], [-45...",106278
5,Ara,Ara,Arae,Altar,2,"[-99.3786, -56.5883, 59]","[-102, -56]",Ara,Altar,None,"[[85267, 85727, 82363, 83081, 83153, 85792, 88...","[[[-110.9653, -60.2645], [-111.4294, -45.7671]...",85792
6,Ari,Aries,Arietis,Ram,0,"[39.5412, 20.7923, 61]","[42, 22]",Aries,Ram,None,"[[8832, 8903, 9884, 13209]]","[[[31.6652, 10.5144], [26.6557, 10.5432], [26....",9884
7,Aur,Auriga,Aurigae,Charioteer,0,"[91, 42, 88]","[82.5, 37]",Auriga,Charioteer,None,"[[25428, 23015, 23767, 24608, 28360, 28380, 25...","[[[69.4869, 30.9219], [69.5738, 36.2547], [72....",24608
8,Boo,Boötes,Boötis,Herdsman,0,"[-140, 31, 108]","[-136.5, 35]",Boötes,Herdsman,None,"[[69673, 72105, 74666, 73555, 71075, 71053, 69...","[[[-132.2185, 7.5254], [-155.9362, 7.3606], [-...",69673
9,CMa,Canis Major,Canis Majoris,Great Dog,0,"[102.4362, -22.1403, 54]","[97.5, -26]",Canis Major,Greater Dog,None,"[[30324, 32349, 34444, 33579, 34444, 35904], [...","[[[93.2156, -11.0302], [111.9734, -11.2521], [...",32349


In [429]:
def choose_meaning(row):
    if row["meaning_1"] == row["name"]:
        if row["meaning_3"]:
            return row["meaning_3"]
        elif row["meaning_2"]:
            return row["meaning_2"]

    return row["meaning_1"]

constellations["meaning"] = constellations.apply(choose_meaning, axis=1) 

constellations = constellations[["id", "name", "gen", "meaning", "rank", "focus", "label", "boundaries", "lines_hip", "brightest_hip" ]]
constellations


,id,name,gen,meaning,rank,focus,label,boundaries,lines_hip,brightest_hip
0,And,Andromeda,Andromedae,Chained Maiden,0,"[-350, 37, 115]","[0.75, 43]","[[[-15.5347, 35.1682], [-15.6571, 53.168], [-8...","[[677, 3092, 5447, 9640], [113726, 116631, 116...",5447
1,Ant,Antlia,Antliae,Air Pump,2,"[154.1067, -32.4836, 60]","[156, -36]","[[[141.9043, -24.5425], [141.7716, -37.292], [...","[[53502, 51172, 46515]]",51172
2,Aps,Apus,Apodis,Bird of Paradise,2,"[-118, -75, 47]","[-120, -74]","[[[-150.8889, -83.1201], [-83.134, -82.4583], ...","[[72370, 81065], [80047, 81852, 81065]]",72370
3,Aql,Aquila,Aquilae,Eagle,0,"[-64.9946, 3.4109, 64]","[-69, 8]","[[[-79.6498, 0.1155], [-79.6738, 2.1153], [-75...","[[98036, 97649, 97278, 95501, 93805, 95501, 93...",97649
4,Aqr,Aquarius,Aquarii,Water Bearer,1,"[-25, -11, 115]","[-22.5, -5]","[[[-50.4012, 0.4362], [-50.4201, 2.4361], [-45...","[[102618, 106278, 109074, 110395, 110960, 1114...",106278
5,Ara,Ara,Arae,Altar,2,"[-99.3786, -56.5883, 59]","[-102, -56]","[[[-110.9653, -60.2645], [-111.4294, -45.7671]...","[[85267, 85727, 82363, 83081, 83153, 85792, 88...",85792
6,Ari,Aries,Arietis,Ram,0,"[39.5412, 20.7923, 61]","[42, 22]","[[[31.6652, 10.5144], [26.6557, 10.5432], [26....","[[8832, 8903, 9884, 13209]]",9884
7,Aur,Auriga,Aurigae,Charioteer,0,"[91, 42, 88]","[82.5, 37]","[[[69.4869, 30.9219], [69.5738, 36.2547], [72....","[[25428, 23015, 23767, 24608, 28360, 28380, 25...",24608
8,Boo,Boötes,Boötis,Herdsman,0,"[-140, 31, 108]","[-136.5, 35]","[[[-132.2185, 7.5254], [-155.9362, 7.3606], [-...","[[69673, 72105, 74666, 73555, 71075, 71053, 69...",69673
9,CMa,Canis Major,Canis Majoris,Great Dog,0,"[102.4362, -22.1403, 54]","[97.5, -26]","[[[93.2156, -11.0302], [111.9734, -11.2521], [...","[[30324, 32349, 34444, 33579, 34444, 35904], [...",32349


# IAU named stars

In [196]:
IAU_URL = "file:///home/sebl/code/etoile/notebooks/IAU-Catalog of Star Names (always up to date).csv"
iau_named = pd.read_csv(IAU_URL).sort_values("proper names")
iau_named = iau_named[iau_named["HIP"].notna()]

# important fields are HIP, proper names, Simbad spelling(for ascii name), Bayer ID and constellation
iau_named = iau_named

print(len(iau_named))
iau_named


506


,proper names,NEC+,Designation,HIP,Bayer ID,Simbad spelling,Constellation,Origin,Language,Reference,Date of Adoption
109,Acamar,920,* tet Eri,13847.0,θ1 Eri,Acamar,Eri,From its name for the Arabic Almagest period a...,Arabic,"Kunitzsch, Paul; Smart, Tim (2006). A Dictiona...",2016/07/20
317,Achernar,499,HR 472,7588.0,α Eri,Achernar,Eri,"Arabic name آخر النهر (āḫiru ʾn-nahr) meaning,...",Arabic,"Kunitzsch, Paul; Smart, Tim (2006). A Dictiona...",2016/06/30
131,Achird,257,HR 219,3821.0,η Cas,Achird,Cas,The proper name Achird was apparently first ap...,Contemporary,--,2017/09/05
239,Acrab,"5,996",* bet Sco,78820.0,β1 Sco,Acrab,Sco,Arabic name al-'Aqrab for the constellation Sc...,Arabic,"Kunitzsch, P. (1959) Arabische Sternnamen in E...",2016/08/21
323,Acrux,"4,739",* alf Cru,60718.0,α Cru,Acrux,Cru,A modern creation (1938) that is derived from ...,Contemporary,"MacKworth, P.H. et al. (1938) ‘The Air Almanac...",2016/07/20
329,Acubens,"3,548",HR 3572,44066.0,α Cnc,Acubens,Cnc,Applied with various spellings since medieval ...,Arabic,"Kunitzsch, Paul; Smart, Tim (2006). A Dictiona...",2016/07/20
144,Adhafera,"4,006",HR 4031,50335.0,ζ Leo,Adhafera,Leo,From the name for the Arabic Almagest period a...,Arabic,"Kunitzsch, Paul; Smart, Tim (2006). A Dictiona...",2016/07/20
172,Adhara,"2,605",HR 2618,33579.0,ε CMa,Adhara,CMa,Applied in recent times from the Arabic asteri...,Arabic,"Kunitzsch, Paul; Smart, Tim (2006). A Dictiona...",2016/08/21
55,Adhil,420,HR 390,6411.0,ξ And,Adhil,And,"From the Arabic word al-dhail, ""the hem of a r...",Arabic,"Kunitzsch, Paul; Smart, Tim (2006). A Dictiona...",2016/08/21
159,Ain,"1,449",HR 1409,20889.0,ε Tau,Ain,Tau,From an abbreviation of the name for the Arabi...,Arabic,"Kunitzsch, Paul; Smart, Tim (2006). A Dictiona...",2015/12/15


# Misc simbad table queries

In [201]:
tables_query = """
SELECT
  table_name,
  description
FROM TAP_SCHEMA.tables
ORDER BY table_name
"""

tables_df = Simbad.query_tap(tables_query)
tables_df

table_name,description
object,object
allfluxes,"all flux/magnitudes U,B,V,I,J,H,K,u_,g_,r_,i_,z_"
alltypes,all object types concatenated with pipe
author,Author of a bibliographic reference
basic,General data about an astronomical object
biblio,Bibliography
cat,Catalogues name
filter,Description of a flux filter
flux,Magnitude/Flux information about an astronomical object
has_ref,Associations between astronomical objects and their bibliographic references


In [200]:
columns_query = """
SELECT
  table_name,
  column_name,
  datatype,
  unit,
  ucd,
  description
FROM TAP_SCHEMA.columns
WHERE table_name = 'mesDistance'
ORDER BY column_name
"""

basic_columns_df = Simbad.query_tap(columns_query)
# basic_columns_df = Simbad.query_tap(columns_query).show_in_browser(jsviewer=True)

basic_columns_df

table_name,column_name,datatype,unit,ucd,description
object,object,object,object,object,object
mesDistance,bibcode,CHAR,,meta.bib.bibcode,measurement bibcode
mesDistance,dist,DOUBLE,,pos.distance,Distance value
mesDistance,dist_prec,SMALLINT,,,Precision (# of decimal positions) associated with the column dist
mesDistance,mespos,SMALLINT,,stat.rank,Position of a measurement in a list of measurements
mesDistance,method,CHAR,,instr.setup,distance calculation method
mesDistance,minus_err,DOUBLE,,stat.error,minus error
mesDistance,minus_err_prec,SMALLINT,,,Precision (# of decimal positions) associated with the column minus_err
mesDistance,oidref,BIGINT,,meta.record,Object internal identifier
mesDistance,plus_err,DOUBLE,,stat.error,plus error


# Seed stars

In [281]:
from astroquery.simbad import Simbad
from astropy.table import vstack, QTable, Table


In [311]:
MISSING_CONSTELLATION_LINE_STARS_HIP = [
    16228,
    17884,
    29997,
    33694,
    44700,
    109754
]

In [312]:
query = f"""
SELECT
    min(nameid.id) AS name_bad,
    b.otype,
    b.ra as ra_1,
    b.dec as dec_1,
    b.sp_type as spect_1,
    b.plx_value AS plx_1,
    af.B - af.V AS ci_1,
    f.teff,
    d.diameter,
    d.unit,
    di.dist,
    di.unit AS unit_dist,
    ids.ids
FROM basic b
LEFT JOIN ident
  ON b.oid = ident.oidref
LEFT JOIN mesDiameter d
  ON d.oidref = b.oid
 AND d.mespos = 1
LEFT JOIN mesDistance di
  ON di.oidref = b.oid
 AND di.mespos = 1
LEFT JOIN allfluxes AS af
  ON af.oidref = b.oid
LEFT JOIN mesFe_H f
  ON f.oidref = b.oid
 AND f.mespos = 1
LEFT JOIN ident AS nameid
  ON b.oid = nameid.oidref
  AND nameid.id LIKE 'NAME %'
LEFT JOIN ids
  ON ids.oidref = b.oid
WHERE
    ident.id LIKE '* %' OR
    ident.id IN ({", ".join(map(lambda x : f"'HIP {x}'", MISSING_CONSTELLATION_LINE_STARS_HIP))})
GROUP BY b.oid, b.main_id, b.ra, b.dec, b.plx_value, b.plx_err, b.sp_type, b.otype, af.b, af.v, f.teff, ids,ids, d.diameter, d.unit, di.dist, di.unit
ORDER BY main_id
"""

rows = Simbad.query_tap(query)
# rows.show_in_browser(jsviewer=True)
len(rows)


4682

In [446]:
SIMBAD_GREEK_ABBR = {
    "alf": "Alpha", "bet": "Beta", "gam": "Gamma", "del": "Delta", "eps": "Epsilon", "zet": "Zeta", "eta": "Eta", "the": "Theta",
    "iot": "Iota", "kap": "Kappa", "lam": "Lambda", "mu.": "Mu", "nu.": "Nu", "ksi": "Xi", "omi": "Omicron", "pi.": "Pi",
    "rho": "Rho", "sig": "Sigma", "tau": "Tau", "ups": "Upsilon", "phi": "Phi", "khi": "Chi", "psi": "Psi", "ome": "Omega",
}

# SIMBAD sometimes uses these Bayer spellings too.
SIMBAD_GREEK_ABBR.update({"chi": "Chi", "tet": "Theta", "mu": "Mu", "nu": "Nu", "pi": "Pi"})

SIMBAD_GREEK_PATTERN = fr"({"|".join(SIMBAD_GREEK_ABBR).replace(".", "\\.")})"
SIMBAD_GREEK_PATTERN


'(alf|bet|gam|del|eps|zet|eta|the|iot|kap|lam|mu\\.|nu\\.|ksi|omi|pi\\.|rho|sig|tau|ups|phi|khi|psi|ome|chi|tet|mu|nu|pi)'

In [447]:
CONSTELLATION_ABBR_PATTERN = fr"({"|".join(constellations["id"])})"
CONSTELLATION_ABBR_PATTERN

'(And|Ant|Aps|Aql|Aqr|Ara|Ari|Aur|Boo|CMa|CMi|CVn|Cae|Cam|Cap|Car|Cas|Cen|Cep|Cet|Cha|Cir|Cnc|Col|Com|CrA|CrB|Crt|Cru|Crv|Cyg|Del|Dor|Dra|Equ|Eri|For|Gem|Gru|Her|Hor|Hya|Hyi|Ind|LMi|Lac|Leo|Lep|Lib|Lup|Lyn|Lyr|Men|Mic|Mon|Mus|Nor|Oct|Oph|Ori|Pav|Peg|Per|Phe|Pic|PsA|Psc|Pup|Pyx|Ret|Scl|Sco|Sct|Ser|Sex|Sge|Sgr|Tau|Tel|TrA|Tri|Tuc|UMa|UMi|Vel|Vir|Vol|Vul)'

## Filter out non stars

In [313]:
# SIMBAD object-type classification patterns
massive_stars_otype_pattern = r"^(?:\*|Ma[*?]|bC[*?]|sg[*?]|s[*?]r|s[*?]y|s[*?]b|WR[*?]|LBV|N\*\??|Psr)$"
young_stars_otype_pattern = r"^(?:Y\*[O?]|Or\*|TT[*?]|Ae[*?]|out|of\?|HH)$"
main_seq_stars_otype_pattern = r"^(?:MS[*?]|Be[*?]|BS[*?]|SX\*|gD\*|dS\*|BY[*?])$"
evolved_stars_otype_pattern = r"^(?:Ev[*?]|RG\*|RB\?|HS[*?]|HB[*?]|RR[*?]|WV[*?]|Ce[*?]|cC\*|C\*\??|S\*\??|LP[*?]|AB[*?]|Mi[*?]|OH[*?]|pA[*?]|RV[*?]|PN\??|WD[*?]|ELMWD)$"
peculiar_stars_otype_pattern = r"^(?:Pe[*?]|a2[*?]|RC[*?])$"
binary_stars_otype_pattern = r"^(?:\*\*\??|El[*?]|EB[*?]|SB[*?]|RS[*?]|Sy[*?]|XB[*?]|LX[B?]|HX[B?]|CV[*?])$" # No[*?] novae omitted
supernovae_otype_pattern = r"^(?:SN[*?])$"
low_mass_stars_otype_pattern = r"^(?:LM[*?]|BD[*?])$"  # Pl\?? extra-solar planet omitted
variable_stars_otype_pattern = r"^(?:V\*\??|Ir\*|Er[*?]|Ro[*?]|Pu[*?])$"
emline_stars_otype_pattern = r"^(?:Em[*?])$"
kinematic_stars_otype_pattern = r"^(?:PM\*|HV\*)$"

star_otype_pattern = "|".join([
    massive_stars_otype_pattern, young_stars_otype_pattern, main_seq_stars_otype_pattern,
    evolved_stars_otype_pattern, peculiar_stars_otype_pattern, binary_stars_otype_pattern,
    low_mass_stars_otype_pattern, variable_stars_otype_pattern, emline_stars_otype_pattern,
    kinematic_stars_otype_pattern,
])

star_otype_re = re.compile(star_otype_pattern)

def regex_mask_column(col, pattern):
    regex = re.compile(pattern)

    mask = []
    for value in col:
        if value is None:
            mask.append(False)
            continue

        # Handles masked values
        if np.ma.is_masked(value):
            mask.append(False)
            continue

        # Handles bytes columns if present
        if isinstance(value, bytes):
            value = value.decode("utf-8")

        value = str(value).strip()

        mask.append(bool(regex.match(value)))

    return np.array(mask, dtype=bool)

def otype_matches(value, pattern):
    return isinstance(value, str) and re.match(pattern, value) is not None


mask = regex_mask_column(rows["otype"], star_otype_pattern)

stars = rows[mask].copy()
print(len(stars))
stars


4539


name_bad,otype,ra_1,dec_1,spect_1,plx_1,ci_1,teff,diameter,unit,dist,unit_dist,ids
,,deg,deg,,mas,,unit-degK,,,,,
object,object,float64,float64,object,float64,float32,int32,float64,object,float64,object,object
,*,352.92515755764,-21.36946231281,F0V,13.6139,0.30100012,7063,--,,73.454,pc,Gaia DR3 2388538245106801280|* 100 Aqr|BD-22 6141|CD-22 16349|CPD-22 8374|FK5 3885|GC 32714|GCRV 14750|GEN# +1.00221357|GSC 06409-01233|HD 221357|HIC 116118|HIP 116118|HR 8932|PPM 275055|SAO 191970|SKY# 44525|TD1 29900|TYC 6409-1233-1|UBV M 27065|YZ 111 15738|uvby98 100221357|2MASS J23314204-2122101|Gaia DR1 2388538240811705344|WEB 20527|Gaia DR2 2388538245106801280
,**,271.9566666666666,26.101111111111113,,--,--,--,--,,--,,PMSC 18038+2605|** STF 2280|WDS J18078+2606AB|CCDM J18079+2606AB|IRAS 18057+2605|* 100 Her|1RXS J180749.6+260558
,**,271.95652826859,26.101261644319997,A3V,15.6527,0.14700031,8204,--,,63.8867,pc,"TIC 320932914|HIP 88818|Gaia DR3 4579990675911466624|* 100 Her A|ADS 11089 A|AG+26 1797|BD+26 3178A|CCDM J18079+2606A|CEL 4628|CSI+26 3178 1|GC 24721|GCRV 10583|GEN# +1.00166045|HD 166045|HIC 88818|HR 6781|IDS 18038+2605 A|PPM 106809|ROT 2555|SAO 85753|SKY# 32952|UBV 15424|UBV M 22666|YZ 26 8675|2MASS J18074956+2606047|TYC 2095-3224-1|WDS J18078+2606Aa,Ab|PMSC 18038+2605Aab|** STF 2280A|** CHR 67|WDS J18078+2606A|WEB 15050|Gaia DR2 4579990675911466624"
,*,271.9562610383799,26.09733280848,A3V,15.6586,0.13000011,8260,--,,63.863,pc,TIC 320932916|HIP 88817|Gaia DR3 4579990675911464704|* 100 Her B|ADS 11089 B|AG+26 1796|BD+26 3178B|BD+26 3178|CCDM J18079+2606B|CSI+26 3178 2|GC 24720|GCRV 10582|GEN# +1.00166046|HD 166046|HIC 88817|HR 6782|IDS 18038+2605 B|N30 4030|PPM 106808|ROT 2556|SAO 85752|SKY# 32953|UBV 15425|UBV M 22667|YZ 26 8674|uvby98 100166046|2MASS J18074950+2605504|WDS J18078+2606B|TYC 2095-3225-1|** STF 2280B|PMSC 18038+2605B|WEB 15051|Gaia DR2 4579990675911464704
,*,23.7150886622,12.558635872650001,A3V,10.3996,0.19999981,--,--,,96.1575,pc,TIC 47119710|IDS 01295+1203 A|HIP 7364|Gaia DR3 2586131788972205696|2MASS J01345161+1233312|* 100 Psc|ADS 1238 A|AG+12 167|AGKR 1356|BD+11 201|CCDM J01349+1234A|CSI+11 201 2|CSV 100123|GC 1904|GSC 00627-01377|HD 9656|HIC 7364|NSV 553|PPM 117471|RAFGL 4120S|SAO 92521|SKY# 2376|SV* ZI 83|TYC 627-1377-1|YZ 12 458|[ZEH2003] RX J0134.8+1233 1|WDS J01349+1234A|Gaia DR2 2586131788972205696
,*,272.22026609551995,20.04523297864,A8III,9.5493,0.15999985,8581,--,,104.72,pc,TIC 296081957|HIP 88899|Gaia DR3 4527896021146097408|PLX 4162|* 101 Her|AG+20 1854|BD+20 3675|GC 24743|GCRV 10597|GEN# +1.00166230|GSC 01575-01960|HD 166230|HIC 88899|HR 6794|IRAS 18067+2002|PPM 106831|ROT 2562|SAO 85770|SKY# 32992|TD1 21768|TYC 1575-1960-1|UBV 15440|UBV M 22686|YZ 0 1128|YZ 20 6319|uvby98 100166230|2MASS J18085285+2002426|PLX 4162.00|WEB 15079|Gaia DR2 4527896021146097408
,*,23.943509724190005,14.661421458280001,B9.5III,2.6376,-0.041999817,10831,--,,379.133,pc,TIC 405374293|HIP 7436|Gaia DR3 2588869916522870144|2MASS J01354643+1439412|* 101 Psc|AG+14 130|BD+13 240|CSV 100124|GC 1929|GCRV 897|GEN# +1.00009766|GSC 00627-00535|HD 9766|HIC 7436|HR 455|N30 318|NSV 559|PPM 117482|ROT 221|SAO 92530|SKY# 2394|SV* ZI 84|TYC 627-535-1|UBV 1651|UBV M 8289|YPAC 278|YZ 14 464|uvby98 100009766|ALS 16404|Gaia DR1 2588869912227223424|WEB 1591|Gaia DR2 2588869916522870144
,PM*,74.9346751193,15.91673241203,F5V,22.3771,0.47299957,6600,--,,44.6885,pc,TIC 389121767|HIP 23214|Gaia DR3 3393284752392701312|2MASS J04594432+1555002|Cl* Melotte 25 S 44|* 101 Tau|AG+15 426|AGKR 4423|BD+15 713|CCDM J04598+1554A|CSI+15 713 1|CSI+15 713 3|GC 6085|GCRV 2950|GEN# +5.20250128|GSC 01281-00792|HD 31845|HIC 23214|IDS 04540+1546 A|JP11 4928|N30 1065|PMC 90-93 2277|PPM 120477|ROT 3907|SAO 94248|SKY# 7804|TD1 3805|TYC 1281-792-1|UBV 4795|UBV M 41334|YZ 15 1367|uvby98 520250128|Cl Melotte 25 128|1RXS J045944.7+155517|WDS J04597+1555A|** BUP 73A|[RSP2011] 618|WEB 4509|Gaia DR2 3393284752392701312


In [336]:
simbad = stars.to_pandas().copy()

def find_first(r, s):
    match = re.search(r, s)

    if match:
        return match.group(1).strip()

    return None

simbad["hip"] = simbad["ids"].apply(lambda x : find_first(r"\bHIP\s+([^|]+)(?=\||$)", x))
simbad["hd"] = simbad["ids"].apply(lambda x : find_first(r"\bHD\s+([^|]+)(?=\||$)", x))
simbad["gaia_dr3"] = simbad["ids"].apply(lambda x : find_first(r"\bGaia\sDR3\s+([^|]+)(?=\||$)", x))
simbad["gaia_dr2"] = simbad["ids"].apply(lambda x : find_first(r"\bGaia\sDR2\s+([^|]+)(?=\||$)", x))
simbad["gl"] = simbad["ids"].apply(lambda x : find_first(r"\b(?:GJ|Gl|GL|Gliese)\s+([^|]+)(?=\||$)", x))

In [448]:
# Bayer entries:
# * alf CMa
# * alf01 CMa
# * alf CMa A
# * alf01 CMa A
BAYER_ROOT_PATTERN = fr"(?:{SIMBAD_GREEK_PATTERN}|[A-Za-z]{{1,3}})"

bayer_re = re.compile(
    fr"^\*\s+(?P<by>{BAYER_ROOT_PATTERN})(?P<ss>\d+)?\s+(?P<con>{CONSTELLATION_ABBR_PATTERN})(?:\s+(?P<component>[A-Z][A-Za-z0-9]*))?$"
)

# Flamsteed entries:
# * 9 CMa
# 9 CMa
flamsteed_re = re.compile(
    fr"^(?:\*\s+)?(?P<number>\d+)\s+(?P<con>{CONSTELLATION_ABBR_PATTERN})(?:\s+(?P<component>[A-Z][A-Za-z0-9]*))?$"
)

# Variable-star entries:
# V* CS Cam
# V* UY Sct
variable_re = re.compile(
    fr"^V\*\s+(?P<v>[A-Z][A-Za-z0-9.]*)\s+(?P<con>{CONSTELLATION_ABBR_PATTERN})(?:\s+(?P<component>[A-Z][A-Za-z0-9]*))?$"
)


def parse_star_ids(ids: str):
    """
    Parse SIMBAD pipe-separated ids.

    Bayer terms split into:
    - by_root: Greek abbreviation only, no superscript (alf, zet, pi., ...)
    - by: SIMBAD Bayer designation, including superscript when present (alf01)
    - by_ss: superscript integer when present
    - by_component: component suffix when present (A, B, AB, ...)

    Flags:
    - is_main: unsuffixed Bayer system id exists (* alf CMa)
    - is_comp: component Bayer id exists (* alf CMa A)
    - main_and_pri: same SIMBAD row carries system id and primary id (A or ss01)
    - ss_is_comp: row gives evidence that superscript is component notation too
                  (superscript id and component id in same row)
    """
    empty = {
        "by": None,
        "by_root": None,
        "by_ss": None,
        "by_component": None,
        "con": None,
        "fl": None,
        "fl_con": None,
        "fl_component": None,
        "v": None,
        "v_con": None,
        "v_component": None,
        "is_main": False,
        "is_comp": False,
        "main_and_pri": False,
        "ss_is_comp": False,
        "bayer_ids": [],
        "flamsteed_ids": [],
        "variable_ids": [],
    }

    if pd.isna(ids):
        return empty

    bayer_ids = []
    flamsteed_ids = []
    variable_ids = []

    for part in [p.strip() for p in str(ids).split("|")]:
        bayer_match = bayer_re.match(part)
        if bayer_match:
            d = bayer_match.groupdict()
            ss = int(d["ss"]) if d["ss"] else None
            bayer_ids.append({
                "id": part,
                "by": f"{d['by']}{ss:02d}" if ss is not None else d["by"],
                "by_root": d["by"],
                "by_ss": ss,
                "con": d["con"],
                "root_key": f"{d['by']} {d['con']}",
                "component": d["component"],
                "is_main": ss is None and d["component"] is None,
                "is_comp": d["component"] is not None,
            })
            continue

        flamsteed_match = flamsteed_re.match(part)
        if flamsteed_match:
            d = flamsteed_match.groupdict()
            flamsteed_ids.append({
                "id": part,
                "fl": int(d["number"]),
                "con": d["con"],
                "component": d["component"],
            })
            continue

        variable_match = variable_re.match(part)
        if variable_match:
            d = variable_match.groupdict()
            variable_ids.append({
                "id": part,
                "v": d["v"],
                "con": d["con"],
                "component": d["component"],
            })

    first_bayer = bayer_ids[0] if bayer_ids else {}
    first_flamsteed = flamsteed_ids[0] if flamsteed_ids else {}
    first_variable = variable_ids[0] if variable_ids else {}
    main_terms = [x for x in bayer_ids if x["is_main"]]
    comp_terms = [x for x in bayer_ids if x["is_comp"]]
    ss_terms = [x for x in bayer_ids if x["by_ss"] is not None]
    primary_terms = [x for x in bayer_ids if x["component"] == "A" or x["by_ss"] == 1]

    out = empty.copy()
    out.update({
        "by": first_bayer.get("by"),
        "by_root": first_bayer.get("by_root"),
        "by_ss": first_bayer.get("by_ss"),
        "by_component": first_bayer.get("component"),
        "con": first_bayer.get("con") or first_flamsteed.get("con") or first_variable.get("con"),
        "fl": first_flamsteed.get("fl"),
        "fl_con": first_flamsteed.get("con"),
        "fl_component": first_flamsteed.get("component"),
        "v": first_variable.get("v"),
        "v_con": first_variable.get("con"),
        "v_component": first_variable.get("component"),
        "is_main": bool(main_terms),
        "is_comp": bool(comp_terms),
        "main_and_pri": bool(main_terms and primary_terms),
        "ss_is_comp": bool(ss_terms and comp_terms),
        "bayer_ids": bayer_ids,
        "flamsteed_ids": flamsteed_ids,
        "variable_ids": variable_ids,
    })
    return out


def add_bayer_group_flags(df):
    df = df.copy()
    parsed = pd.DataFrame(df["ids"].apply(parse_star_ids).tolist(), index=df.index)
    df = pd.concat([df.drop(columns=[c for c in parsed.columns if c in df.columns], errors="ignore"), parsed], axis=1)

    df["bayer_root_key"] = np.where(
        df["by_root"].notna() & df["con"].notna(),
        df["by_root"].astype(str) + " " + df["con"].astype(str),
        None,
    )
    df["flamsteed_key"] = np.where(
        df["fl"].notna() & df["fl_con"].notna(),
        df["fl"].astype("Int64").astype(str) + " " + df["fl_con"].astype(str),
        None,
    )
    df["variable_key"] = np.where(
        df["v"].notna() & df["v_con"].notna(),
        df["v"].astype(str) + " " + df["v_con"].astype(str),
        None,
    )

    # Parent exists only when unsuperscripted, componentless Bayer id exists somewhere: * tau Hyi.
    # If parent missing, superscripted ids are independent main systems: * tau01 Hyi, * tau02 Hyi.
    parent_keys = {
        x["root_key"]
        for ids in df["bayer_ids"]
        for x in ids
        if x["by_ss"] is None and x["component"] is None
    }

    def bayer_group_key(ids):
        if not ids:
            return None
        x = ids[0]
        if x["by_ss"] is not None and x["root_key"] not in parent_keys:
            return f"{x['by']} {x['con']}"
        return x["root_key"]

    def bayer_is_main(ids):
        return any(
            x["component"] is None and (x["by_ss"] is None or x["root_key"] not in parent_keys)
            for x in ids
        )

    def bayer_is_primary(ids):
        return any(x["component"] == "A" for x in ids)

    df["bayer_key"] = df["bayer_ids"].apply(bayer_group_key)

    # Inherit Bayer from parent system when child only has same Flamsteed/V* component id.
    # Example: parent "* tau02 Cap|* 14 Cap" and child "* 14 Cap A" => inferred "tau02 Cap A".
    for fallback_key in ["flamsteed_key", "variable_key"]:
        for key, g in df[df[fallback_key].notna()].groupby(fallback_key, dropna=True):
            parent = g[g["bayer_key"].notna() & g["by_component"].isna()]
            if parent.empty:
                continue
            # Prefer exact superscripted Bayer parent over broader unsuperscripted Bayer root.
            # Example: 14 Cap parent has tau02 Cap; components should become tau02 Cap A/B, not tau Cap A/B.
            parent = parent.assign(_has_bayer_ss=parent["by_ss"].notna())
            parent_row = parent.sort_values("_has_bayer_ss", ascending=False).iloc[0]
            child_idx = g[g["bayer_key"].isna() & (g["fl_component"].notna() | g["v_component"].notna())].index
            for idx in child_idx:
                comp = df.at[idx, "fl_component"] if pd.notna(df.at[idx, "fl_component"]) else df.at[idx, "v_component"]
                df.at[idx, "by"] = parent_row["by"]
                df.at[idx, "by_root"] = parent_row["by_root"]
                df.at[idx, "by_ss"] = parent_row["by_ss"]
                df.at[idx, "by_component"] = comp
                df.at[idx, "bayer_root_key"] = parent_row["bayer_root_key"]
                df.at[idx, "bayer_key"] = parent_row["bayer_key"]


    # Superscript means component only when an unsuperscripted parent exists AND SIMBAD co-locates ss + component.
    ss_comp_root_keys = {
        row["bayer_root_key"]
        for _, row in df.iterrows()
        if row["bayer_root_key"] in parent_keys
        and any(x["by_ss"] is not None for x in row["bayer_ids"])
        and any(x["component"] is not None for x in row["bayer_ids"])
    }
    df["ss_is_comp"] = df.apply(
        lambda row: row["bayer_root_key"] in ss_comp_root_keys and pd.notna(row["by_ss"]),
        axis=1,
    )

    # Choose deconflict key: Bayer first, then Flamsteed, then V*. This allows HIP transfer for 103 Psc -> 103 Psc A.
    df["designation_key"] = df["bayer_key"].combine_first(df["flamsteed_key"]).combine_first(df["variable_key"])
    df["designation_kind"] = np.select(
        [df["bayer_key"].notna(), df["flamsteed_key"].notna(), df["variable_key"].notna()],
        ["bayer", "flamsteed", "variable"],
        default=None,
    )

    df["is_main"] = (
        df["bayer_ids"].apply(bayer_is_main)
        | (df["bayer_key"].isna() & df["flamsteed_key"].notna() & df["fl_component"].isna())
        | (df["bayer_key"].isna() & df["flamsteed_key"].isna() & df["variable_key"].notna() & df["v_component"].isna())
    )
    df["is_comp"] = (
        df["bayer_ids"].apply(lambda ids: any(x["component"] is not None for x in ids))
        | df["by_component"].notna()
        | (df["bayer_key"].isna() & df["fl_component"].notna())
        | (df["bayer_key"].isna() & df["flamsteed_key"].isna() & df["v_component"].notna())
    )
    df["is_primary"] = (
        df["bayer_ids"].apply(bayer_is_primary)
        | df["by_component"].eq("A")
        | (df["ss_is_comp"] & df["by_ss"].eq(1))
        | (df["bayer_key"].isna() & df["fl_component"].eq("A"))
        | (df["bayer_key"].isna() & df["flamsteed_key"].isna() & df["v_component"].eq("A"))
    )
    df["main_and_pri"] = df["is_main"] & df["is_primary"]
    return df


def build_designation_primary_crosswalk(df, key_col="designation_key"):
    rows = []
    for key, g in df[df[key_col].notna()].groupby(key_col, dropna=True):
        main = g[g["is_main"]]
        pri = g[g["is_primary"]]
        if pri.empty:
            pri = main
        if pri.empty:
            continue

        pri_row = pri.assign(_comp=pri["by_component"].fillna(pri["fl_component"]).fillna(pri["v_component"]).fillna("")).sort_values(
            ["main_and_pri", "_comp", "by_ss"],
            ascending=[False, True, True],
            na_position="last",
        ).iloc[0]
        main_row = main.iloc[0] if not main.empty else pri_row
        main_hip = main_row.get("hip")
        pri_hip = pri_row.get("hip")

        if main_row.name == pri_row.name:
            category = "main_is_pri"
        elif pd.notna(main_hip) and pd.notna(pri_hip) and str(main_hip) != str(pri_hip):
            category = "main and pri have diff hip"
        elif pd.notna(main_hip) and pd.isna(pri_hip):
            category = "main has hip, pri doesnt"
        elif pd.isna(main_hip) and pd.notna(pri_hip):
            category = "pri has hip, main doesnt"
        elif pd.notna(main_hip) and pd.notna(pri_hip):
            category = "main and pri share hip"
        else:
            category = "no hip"

        rows.append({
            "designation_key": key,
            "designation_kind": first_notna(main_row.get("designation_kind"), pri_row.get("designation_kind")) if "first_notna" in globals() else main_row.get("designation_kind"),
            "main_index": main_row.name,
            "primary_index": pri_row.name,
            "main_hip": main_hip,
            "primary_hip": pri_hip,
            "effective_primary_hip": pri_hip if pd.notna(pri_hip) else main_hip,
            "category": category,
            "main_ids": main_row["ids"],
            "primary_ids": pri_row["ids"],
        })
    return pd.DataFrame(rows)


def build_bayer_primary_crosswalk(df):
    out = build_designation_primary_crosswalk(df[df["bayer_key"].notna()].copy(), key_col="bayer_key")
    if not out.empty:
        out = out.rename(columns={"designation_key": "bayer_key"})
    return out

simbad = add_bayer_group_flags(simbad)
designation_primary = build_designation_primary_crosswalk(simbad)
bayer_primary = build_bayer_primary_crosswalk(simbad)

# QTable.from_pandas(simbad).show_in_browser(jsviewer=True)
simbad


,name_bad,otype,ra_1,dec_1,spect_1,plx_1,ci_1,teff,diameter,unit,dist,unit_dist,ids,hip,hd,gaia_dr3,gaia_dr2,gl,bayer_root_key,bayer_key,is_primary,hip_int,hd_int,gaia_dr3_int,gaia_dr2_int,gl_norm,merged_hip,merged_hd,merged_gaia_dr3,merged_gaia_dr2,merged_gl,flamsteed_key,variable_key,designation_key,designation_kind,by,by_root,by_ss,by_component,con,fl,fl_con,fl_component,v,v_con,v_component,is_main,is_comp,main_and_pri,ss_is_comp,bayer_ids,flamsteed_ids,variable_ids
0,,*,352.925158,-21.369462,F0V,13.6139,0.301,7063,NaN,,73.4540,pc,Gaia DR3 2388538245106801280|* 100 Aqr|BD-22 ...,116118,221357,2388538245106801280,2388538245106801280,None,None,None,False,116118,221357,2388538245106801280,2388538245106801280,<NA>,116118,221357,2388538245106801280,2388538245106801280,<NA>,100 Aqr,None,100 Aqr,flamsteed,None,None,NaN,None,Aqr,100.0,Aqr,None,None,None,None,True,False,False,False,[],"[{'id': '* 100 Aqr', 'fl': 100, 'con': 'Aqr', ...",[]
1,,**,271.956667,26.101111,,NaN,NaN,<NA>,NaN,,NaN,,PMSC 18038+2605|** STF 2280|WDS J18078+2606AB|...,None,None,None,None,None,None,None,False,<NA>,<NA>,<NA>,<NA>,<NA>,88818,166045,4579990675911466624,4579990675911466624,<NA>,100 Her,None,100 Her,flamsteed,None,None,NaN,None,Her,100.0,Her,None,None,None,None,True,False,False,False,[],"[{'id': '* 100 Her', 'fl': 100, 'con': 'Her', ...",[]
2,,**,271.956528,26.101262,A3V,15.6527,0.147,8204,NaN,,63.8867,pc,TIC 320932914|HIP 88818|Gaia DR3 4579990675911...,88818,166045,4579990675911466624,4579990675911466624,None,None,None,True,88818,166045,4579990675911466624,4579990675911466624,<NA>,88818,166045,4579990675911466624,4579990675911466624,<NA>,100 Her,None,100 Her,flamsteed,None,None,NaN,None,Her,100.0,Her,A,None,None,None,False,True,False,False,[],"[{'id': '* 100 Her A', 'fl': 100, 'con': 'Her'...",[]
3,,*,271.956261,26.097333,A3V,15.6586,0.130,8260,NaN,,63.8630,pc,TIC 320932916|HIP 88817|Gaia DR3 4579990675911...,88817,166046,4579990675911464704,4579990675911464704,None,None,None,False,88817,166046,4579990675911464704,4579990675911464704,<NA>,88817,166046,4579990675911464704,4579990675911464704,<NA>,100 Her,None,100 Her,flamsteed,None,None,NaN,None,Her,100.0,Her,B,None,None,None,False,True,False,False,[],"[{'id': '* 100 Her B', 'fl': 100, 'con': 'Her'...",[]
4,,*,23.715089,12.558636,A3V,10.3996,0.200,<NA>,NaN,,96.1575,pc,TIC 47119710|IDS 01295+1203 A|HIP 7364|Gaia DR...,7364,9656,2586131788972205696,2586131788972205696,None,None,None,False,7364,9656,2586131788972205696,2586131788972205696,<NA>,7364,9656,2586131788972205696,2586131788972205696,<NA>,100 Psc,None,100 Psc,flamsteed,None,None,NaN,None,Psc,100.0,Psc,None,None,None,None,True,False,False,False,[],"[{'id': '* 100 Psc', 'fl': 100, 'con': 'Psc', ...",[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4534,,*,115.469059,-72.608255,,22.6130,NaN,<NA>,NaN,,44.2220,pc,Gaia DR2 5263150888430032384|TIC 272086940|Gai...,None,63295B,5263150888430032384,5263150888430032384,None,zet Vol,zet Vol,False,<NA>,<NA>,5263150888430032384,5263150888430032384,<NA>,<NA>,<NA>,5263150888430032384,5263150888430032384,<NA>,None,None,zet Vol,bayer,zet,zet,NaN,B,Vol,NaN,None,None,None,None,None,False,True,False,False,"[{'id': '* zet Vol B', 'by': 'zet', 'by_root':...",[],[]
4535,,Be*,267.513894,48.394152,B6IIInp_sh,3.1524,-0.100,<NA>,NaN,,317.2190,pc,TIC 356239023|AP J17500331+4823391|HIP 87280|G...,87280,162732,1363284299777747584,1363284299777747584,None,z Her,z Her,False,87280,162732,1363284299777747584,1363284299777747584,<NA>,87280,162732,1363284299777747584,1363284299777747584,<NA>,88 Her,V744 Her,z Her,bayer,z,z,NaN,None,Her,88.0,Her,None,V744,Her,None,True,False,False,False,"[{'id': '* z Her', 'by': 'z', 'by_root': 'z', ...","[{'id': '* 88 Her', 'fl': 88, 'con': 'Her', '...","[{'id': 'V* V744 Her', 'v': 'V744', 'con': 'He..."
4536,,Be*,113.462673,-36.33

In [449]:
def flatten_hips(value):
    """Flatten Stellarium/D3 constellation path structures into HIP ints."""
    out = []

    def walk(x):
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return
        if isinstance(x, (list, tuple, set)):
            for item in x:
                walk(item)
            return
        try:
            out.append(int(x))
        except (TypeError, ValueError):
            pass

    walk(value)
    return out


constellation_path_hips = sorted({
    hip
    for paths in constellations["lines_hip"].dropna()
    for hip in flatten_hips(paths)
})

simbad_by_hip = simbad[simbad["hip"].notna()].copy()
simbad_by_hip["hip_int"] = pd.to_numeric(simbad_by_hip["hip"], errors="coerce")
simbad_by_hip = simbad_by_hip[simbad_by_hip["hip_int"].notna()].copy()
simbad_by_hip["hip_int"] = simbad_by_hip["hip_int"].astype(int)

constellation_hip_matches = (
    pd.DataFrame({"hip_int": constellation_path_hips})
    .merge(
        simbad_by_hip[[
            "hip_int", "ids", "otype", "by", "by_root", "by_ss", "by_component", "con",
            "fl", "fl_component", "v", "v_component",
            "is_main", "is_comp", "main_and_pri", "ss_is_comp", "is_primary", "bayer_key",
        ]],
        on="hip_int",
        how="left",
        indicator=True,
    )
)

constellation_hip_matches["mapped_to"] = np.select(
    [
        constellation_hip_matches["is_main"].eq(True) & constellation_hip_matches["is_primary"].eq(True),
        constellation_hip_matches["is_main"].eq(True),
        constellation_hip_matches["is_primary"].eq(True),
        constellation_hip_matches["is_comp"].eq(True),
        constellation_hip_matches["_merge"].eq("left_only"),
    ],
    ["main_and_primary", "system/main", "primary", "component", "not_in_simbad_seed"],
    default="single_or_flamsteed_only",
)

constellation_hip_matches.sort_values(["mapped_to", "hip_int"]).reset_index(drop=True)

# missing constellation line stars
# 16228
# 17884
# 29997
# 33694
# 44700
# 109754

,hip_int,ids,otype,by,by_root,by_ss,by_component,con,fl,fl_component,v,v_component,is_main,is_comp,main_and_pri,ss_is_comp,is_primary,bayer_key,_merge,mapped_to
0,5742,TIC 16879871|HIP 5742|Gaia DR3 294090536905167...,SB*,phi,phi,NaN,None,Psc,85.0,None,None,None,True,True,True,False,True,phi Psc,both,main_and_primary
1,9640,TIC 292057658|HIC 9640|HIP 9640|AG+42 213|*...,*,gam,gam,NaN,None,And,57.0,A,None,None,True,True,True,False,True,gam And,both,main_and_primary
2,10826,TIC 332890609|4XMM J021920.7-025841|HIP 10826|...,Mi*,omi,omi,NaN,None,Cet,68.0,None,None,None,True,True,True,False,True,omi Cet,both,main_and_primary
3,12413,TIC 142268253|WISEA J023948.09-425330.2|HIP 12...,SB*,s,s,NaN,None,Eri,NaN,None,None,None,True,True,True,False,True,s Eri,both,main_and_primary
4,13531,TIC 285667894|HIP 13531|Gaia DR3 4411616932727...,SB*,tau,tau,NaN,None,Per,18.0,None,None,None,True,True,True,False,True,tau Per,both,main_and_primary
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
740,116805,TIC 176945734|Gaia DR3 1926476042681491712|WDS...,PM*,kap,kap,NaN,None,And,19.0,None,None,None,True,False,False,False,False,kap And,both,system/main
741,116928,TIC 419936581|Gaia DR3 2646511813609323008|PLX...,PM*,lam,lam,NaN,None,Psc,18.0,None,None,None,True,False,False,False,False,lam Psc,both,system/main
742,117452,TIC 33982930|Gaia DR3 2328250059757795072|WISE...,PM*,del,del,NaN,None,Scl,NaN,None,None,None,True,False,False,False,False,del Scl,both,system/main
743,118268,TIC 401396260|Gaia DR3 2745491430891511168|2E ...,SB*,ome,ome,NaN,None,Psc,28.0,None,None,None,True,False,False,False,False,ome Psc,both,system/main


In [450]:
# HIP + identifier deconflict map: any system/main ids resolve to primary ids when known.
# Uses Bayer first, then Flamsteed, then V* designation groups.
designation_id_crosswalk = designation_primary.copy()
designation_id_crosswalk = designation_id_crosswalk[designation_id_crosswalk["effective_primary_hip"].notna()].copy()

def numeric_id(value):
    if pd.isna(value):
        return None
    text = str(value).strip().replace(",", "").replace(" ", "")
    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]
    return int(text) if re.fullmatch(r"\d+", text) else None

def nullable_int_series(series):
    return pd.Series([numeric_id(x) for x in series], index=series.index, dtype="Int64")

numeric_hip = numeric_id

# First make numeric identifier columns. Avoid pd.to_numeric for Gaia source ids; float coercion loses precision.
simbad["hip_int"] = nullable_int_series(simbad["hip"])
for col in ["hd", "gaia_dr3", "gaia_dr2"]:
    simbad[f"{col}_int"] = nullable_int_series(simbad[col])

# GL/GJ identifiers can contain suffixes; keep string but normalize whitespace.
simbad["gl_norm"] = simbad["gl"].astype("string").str.strip()

id_cols = {
    "hip": "hip_int",
    "hd": "hd_int",
    "gaia_dr3": "gaia_dr3_int",
    "gaia_dr2": "gaia_dr2_int",
    "gl": "gl_norm",
}

# Each designation group gets one effective id value, preferring primary row, then main row, then any group row.
bayer_identifier_crosswalk_rows = []
for _, row in designation_primary.iterrows():
    group = simbad[simbad["designation_key"].eq(row["designation_key"])]
    main = simbad.loc[row["main_index"]] if row["main_index"] in simbad.index else pd.Series(dtype=object)
    primary = simbad.loc[row["primary_index"]] if row["primary_index"] in simbad.index else pd.Series(dtype=object)

    out = {
        "designation_key": row["designation_key"],
        "designation_kind": row.get("designation_kind"),
        "main_index": row["main_index"],
        "primary_index": row["primary_index"],
        "category": row["category"],
    }

    for label, col in id_cols.items():
        candidates = [primary.get(col), main.get(col)] + group[col].dropna().tolist()
        value = next((x for x in candidates if pd.notna(x)), pd.NA)
        out[f"merged_{label}"] = value

    bayer_identifier_crosswalk_rows.append(out)

designation_identifier_crosswalk = pd.DataFrame(bayer_identifier_crosswalk_rows)
bayer_identifier_crosswalk = designation_identifier_crosswalk[designation_identifier_crosswalk["designation_kind"].eq("bayer")].copy()

# HIP lookup for constellation lines and system rows. Only parent/main HIPs map to primary HIP.
# Do not overwrite secondary component HIPs.
hip_to_primary_hip = {}
for _, row in designation_identifier_crosswalk.iterrows():
    target = numeric_hip(row["merged_hip"])
    if target is None:
        continue
    for idx_col in ["main_index", "primary_index"]:
        idx = row.get(idx_col)
        if idx in simbad.index:
            source = numeric_hip(simbad.loc[idx, "hip_int"])
            if source is not None:
                hip_to_primary_hip[source] = target

# Write merged ids back to every row in group. Non-Bayer rows keep own ids.
for label, col in id_cols.items():
    simbad[f"merged_{label}"] = simbad[col]

for _, row in designation_identifier_crosswalk.iterrows():
    # Move parent/system ids onto primary row too. Leave secondary components with their own ids/missing ids.
    target_indices = [idx for idx in [row.get("main_index"), row.get("primary_index")] if idx in simbad.index]
    for label in id_cols:
        value = row[f"merged_{label}"]
        if pd.notna(value):
            simbad.loc[target_indices, f"merged_{label}"] = value

simbad["merged_hip"] = nullable_int_series(simbad["merged_hip"])
simbad["merged_hd"] = nullable_int_series(simbad["merged_hd"])
simbad["merged_gaia_dr3"] = nullable_int_series(simbad["merged_gaia_dr3"])
simbad["merged_gaia_dr2"] = nullable_int_series(simbad["merged_gaia_dr2"])
simbad["merged_gl"] = simbad["merged_gl"].astype("string")

constellation_hip_matches["merged_hip"] = (
    constellation_hip_matches["hip_int"]
    .map(hip_to_primary_hip)
    .fillna(constellation_hip_matches["hip_int"])
    .astype(int)
)

designation_identifier_crosswalk.sort_values(["category", "designation_key"])





,designation_key,designation_kind,main_index,primary_index,category,merged_hip,merged_hd,merged_gaia_dr3,merged_gaia_dr2,merged_gl
679,3 Cen,flamsteed,1071,1072,main and pri have diff hip,67669,120709,6170485544575679104,6170485544575679104,<NA>
826,35 Sex,flamsteed,950,951,main and pri have diff hip,52452,92841,3858258736489909632,3858258736489909632,<NA>
896,39 Boo,flamsteed,1038,1039,main and pri have diff hip,72524,131041,1590752742798813824,1590752742798813824,<NA>
1195,53 Aqr,flamsteed,1410,1411,main and pri have diff hip,110778,212698,2595463992699783424,2595463996992115840,859 A
1229,55 Eri,flamsteed,1454,1455,main and pri have diff hip,21986,30021,3185880919107361024,3185880919107361024,<NA>
...,...,...,...,...,...,...,...,...,...,...
3165,psi Psc,bayer,3956,1802,"pri has hip, main doesnt",5131,6456,2790497534190737152,2790497534190737152,<NA>
3250,s Vel,bayer,4125,4126,"pri has hip, main doesnt",51561,91355,5367389229311297280,5367389229311297280,<NA>
3288,sig UMa,bayer,4122,4061,"pri has hip, main doesnt",44857,77800,1068971975056597632,1068971975056597632,335
3409,tet Ori,bayer,4289,4216,"pri has hip, main doesnt",26220,37020,3017364132050194688,3017364132050194688,<NA>


In [451]:
# Additional source data for position, distance/parallax, spectral type, and color.
# Source suffixes:
# _1 = SIMBAD
# _2 = Gaia DR3
# _3 = Gaia DR2
# _4 = HYG
from astroquery.gaia import Gaia

# Gaia archive sometimes returns HTTP 500/null. Keep Gaia enrichment optional/fault-tolerant.
RUN_GAIA_QUERIES = False
GAIA_CHUNK_SIZE = 100


def chunks(values, size=GAIA_CHUNK_SIZE):
    values = [str(v) for v in pd.Series(values).dropna().astype("Int64").unique()]
    for i in range(0, len(values), size):
        yield values[i:i + size]


def empty_gaia_dr3_source_data():
    return pd.DataFrame(columns=[
        "gaia_dr3_int", "ra_2", "dec_2", "plx_2", "dist_2", "ci_2", "teff_2",
        "g_mag_2", "bp_mag_2", "rp_mag_2", "bp_rp_2",
    ])


def empty_gaia_dr2_source_data():
    return pd.DataFrame(columns=[
        "gaia_dr2_int", "ra_3", "dec_3", "plx_3", "dist_3", "ci_3", "teff_3",
        "g_mag_3", "bp_mag_3", "rp_mag_3", "bp_rp_3",
    ])


def query_gaia_dr3(source_ids, chunk_size=GAIA_CHUNK_SIZE):
    if not RUN_GAIA_QUERIES:
        return empty_gaia_dr3_source_data()

    frames = []
    for ids in chunks(source_ids, chunk_size):
        query = f"""
        SELECT
            s.source_id AS gaia_dr3_int,
            s.ra AS ra_2,
            s.dec AS dec_2,
            s.parallax AS plx_2,
            CASE WHEN s.parallax > 0 THEN 1000.0 / s.parallax ELSE NULL END AS dist_2,
            s.phot_bp_mean_mag - s.phot_rp_mean_mag AS ci_2,
            s.teff_gspphot AS teff_2,
            s.phot_g_mean_mag AS g_mag_2,
            s.phot_bp_mean_mag AS bp_mag_2,
            s.phot_rp_mean_mag AS rp_mag_2,
            s.bp_rp AS bp_rp_2
        FROM gaiadr3.gaia_source AS s
        WHERE s.source_id IN ({','.join(ids)})
        """
        try:
            frames.append(Gaia.launch_job(query).get_results().to_pandas())
        except Exception as e:
            print(f"Gaia DR3 chunk failed ({len(ids)} ids): {type(e).__name__}: {e}")
            continue

    return pd.concat(frames, ignore_index=True) if frames else empty_gaia_dr3_source_data()


def query_gaia_dr2(source_ids, chunk_size=GAIA_CHUNK_SIZE):
    if not RUN_GAIA_QUERIES:
        return empty_gaia_dr2_source_data()

    frames = []
    for ids in chunks(source_ids, chunk_size):
        query = f"""
        SELECT
            s.source_id AS gaia_dr2_int,
            s.ra AS ra_3,
            s.dec AS dec_3,
            s.parallax AS plx_3,
            CASE WHEN s.parallax > 0 THEN 1000.0 / s.parallax ELSE NULL END AS dist_3,
            s.phot_bp_mean_mag - s.phot_rp_mean_mag AS ci_3,
            s.teff_val AS teff_3,
            s.phot_g_mean_mag AS g_mag_3,
            s.phot_bp_mean_mag AS bp_mag_3,
            s.phot_rp_mean_mag AS rp_mag_3,
            s.bp_rp AS bp_rp_3
        FROM gaiadr2.gaia_source AS s
        WHERE s.source_id IN ({','.join(ids)})
        """
        try:
            frames.append(Gaia.launch_job(query).get_results().to_pandas())
        except Exception as e:
            print(f"Gaia DR2 chunk failed ({len(ids)} ids): {type(e).__name__}: {e}")
            continue

    return pd.concat(frames, ignore_index=True) if frames else empty_gaia_dr2_source_data()


# One row per merged target id for constellation deconflict work. Prefer HIP, but keep non-HIP targets distinct.
def merged_source_key(row):
    for col in ["merged_hip", "merged_gaia_dr3", "merged_gaia_dr2", "merged_hd", "merged_gl"]:
        value = row.get(col)
        if pd.notna(value):
            return f"{col}:{value}"
    return f"row:{row.name}"

source_base = simbad.copy()
source_base["source_key"] = source_base.apply(merged_source_key, axis=1)
star_source_data = source_base.drop_duplicates("source_key").copy()

# Drop raw parsed id columns before renaming merged ids; otherwise duplicate labels break merge().
star_source_data = star_source_data.drop(
    columns=["hip_int", "hd_int", "gaia_dr3_int", "gaia_dr2_int", "gl_norm"],
    errors="ignore",
)
star_source_data = star_source_data.rename(columns={
    "merged_hip": "hip_int",
    "merged_hd": "hd_int",
    "merged_gl": "gl_id",
    "merged_gaia_dr3": "gaia_dr3_int",
    "merged_gaia_dr2": "gaia_dr2_int",
})

star_source_data = star_source_data[[
    "source_key", "hip_int", "hd_int", "gl_id", "gaia_dr3_int", "gaia_dr2_int",
    "ids", "otype", "by", "con", "fl", "v",
    "ra_1", "dec_1", "plx_1", "dist", "unit_dist", "spect_1", "ci_1", "teff",
]].rename(columns={
    "dist": "dist_1",
    "teff": "teff_1",
})

# SIMBAD mesDistance can have different units. Keep only parsecs for exported dist.
star_source_data["dist_1_pc"] = np.where(
    star_source_data["unit_dist"].astype(str).str.strip().str.lower().eq("pc"),
    star_source_data["dist_1"],
    np.nan,
)

# Gaia source data. Run these only when network/TAP time is acceptable.
gaia_dr3_source_data = query_gaia_dr3(star_source_data["gaia_dr3_int"])
gaia_dr2_source_data = query_gaia_dr2(star_source_data["gaia_dr2_int"])

# HYG source data.
hyg_source_data = hyg.copy()
hyg_source_data["hip_int"] = pd.to_numeric(hyg_source_data["hip"], errors="coerce").astype("Int64")
hyg_source_data = hyg_source_data[["hip_int", "ra", "dec", "dist", "spect", "ci"]].rename(columns={
    "ra": "ra_hours_4",
    "dec": "dec_4",
    "dist": "dist_4",
    "spect": "spect_4",
    "ci": "ci_4",
})
hyg_source_data["ra_4"] = hyg_source_data["ra_hours_4"] * 15.0
hyg_source_data["plx_4"] = np.where(hyg_source_data["dist_4"] > 0, 1000.0 / hyg_source_data["dist_4"], np.nan)

gaia_dr3_source_data = gaia_dr3_source_data[gaia_dr3_source_data["gaia_dr3_int"].notna()].drop_duplicates("gaia_dr3_int")
gaia_dr2_source_data = gaia_dr2_source_data[gaia_dr2_source_data["gaia_dr2_int"].notna()].drop_duplicates("gaia_dr2_int")
hyg_source_data = hyg_source_data[hyg_source_data["hip_int"].notna()].drop_duplicates("hip_int")

star_source_data = star_source_data.merge(gaia_dr3_source_data, on="gaia_dr3_int", how="left")
star_source_data = star_source_data.merge(gaia_dr2_source_data, on="gaia_dr2_int", how="left")
star_source_data = star_source_data.merge(hyg_source_data, on="hip_int", how="left")

# Preferred working fields, not final classification.
def coalesce_columns(df, cols):
    out = pd.Series(pd.NA, index=df.index, dtype="object")
    for col in cols:
        if col in df.columns:
            out = out.where(out.notna(), df[col])
    return out

star_source_data["best_ra"] = coalesce_columns(star_source_data, ["ra_2", "ra_3", "ra_1", "ra_4"])
star_source_data["best_dec"] = coalesce_columns(star_source_data, ["dec_2", "dec_3", "dec_1", "dec_4"])
star_source_data["best_dist"] = coalesce_columns(star_source_data, ["dist_2", "dist_3", "dist_4", "dist_1_pc"])
star_source_data["best_plx"] = coalesce_columns(star_source_data, ["plx_2", "plx_3", "plx_1", "plx_4"])
star_source_data["best_ci"] = coalesce_columns(star_source_data, ["ci_1", "ci_2", "ci_3", "ci_4"])
star_source_data["best_teff"] = coalesce_columns(star_source_data, ["teff_2", "teff_3", "teff_1"])
star_source_data["best_spect"] = coalesce_columns(star_source_data, ["spect_1", "spect_4"])

star_source_data





,source_key,hip_int,hd_int,gl_id,gaia_dr3_int,gaia_dr2_int,ids,otype,by,con,fl,v,ra_1,dec_1,plx_1,dist_1,unit_dist,spect_1,ci_1,teff_1,dist_1_pc,ra_2,dec_2,plx_2,dist_2,ci_2,teff_2,g_mag_2,bp_mag_2,rp_mag_2,bp_rp_2,ra_3,dec_3,plx_3,dist_3,ci_3,teff_3,g_mag_3,bp_mag_3,rp_mag_3,bp_rp_3,ra_hours_4,dec_4,dist_4,spect_4,ci_4,ra_4,plx_4,best_ra,best_dec,best_dist,best_plx,best_ci,best_teff,best_spect
0,merged_hip:116118,116118,221357,<NA>,2388538245106801280,2388538245106801280,Gaia DR3 2388538245106801280|* 100 Aqr|BD-22 ...,*,None,Aqr,100.0,None,352.925158,-21.369462,13.6139,73.4540,pc,F0V,0.301,7063,73.4540,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.528345,-21.369457,76.1615,F0V,0.319,352.925175,13.129994,352.925158,-21.369462,76.1615,13.6139,0.301,7063.0,F0V
1,merged_hip:88818,88818,166045,<NA>,4579990675911466624,4579990675911466624,PMSC 18038+2605|** STF 2280|WDS J18078+2606AB|...,**,None,Her,100.0,None,271.956667,26.101111,NaN,NaN,,,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.130434,26.101280,49.6032,A3V,0.158,271.956510,20.159990,271.956667,26.101111,49.6032,20.15999,0.158,NaN,
2,merged_hip:88817,88817,166046,<NA>,4579990675911464704,4579990675911464704,TIC 320932916|HIP 88817|Gaia DR3 4579990675911...,*,None,Her,100.0,None,271.956261,26.097333,15.6586,63.8630,pc,A3V,0.130,8260,63.8630,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.130417,26.097340,38.5802,A3V,0.127,271.956255,25.920032,271.956261,26.097333,38.5802,15.6586,0.13,8260.0,A3V
3,merged_hip:7364,7364,9656,<NA>,2586131788972205696,2586131788972205696,TIC 47119710|IDS 01295+1203 A|HIP 7364|Gaia DR...,*,None,Psc,100.0,None,23.715089,12.558636,10.3996,96.1575,pc,A3V,0.200,<NA>,96.1575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.581005,12.558633,231.4815,A3V,0.192,23.715075,4.320000,23.715089,12.558636,231.4815,10.3996,0.2,NaN,A3V
4,merged_hip:88899,88899,166230,<NA>,4527896021146097408,4527896021146097408,TIC 296081957|HIP 88899|Gaia DR3 4527896021146...,*,None,Her,101.0,None,272.220266,20.045233,9.5493,104.7200,pc,A8III,0.160,8581,104.7200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.148018,20.045233,100.8065,A8III,0.176,272.220270,9.919995,272.220266,20.045233,100.8065,9.5493,0.16,8581.0,A8III
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4300,merged_gaia_dr3:5263150888430032384,<NA>,<NA>,<NA>,5263150888430032384,5263150888430032384,Gaia DR2 5263150888430032384|TIC 272086940|Gai...,*,zet,Vol,NaN,None,115.469059,-72.608255,22.6130,44.2220,pc,,NaN,<NA>,44.2220,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,115.469059,-72.608255,44.222,22.613,NaN,NaN,
4301,merged_hip:87280,87280,162732,<NA>,1363284299777747584,1363284299777747584,TIC 356239023|AP J17500331+4823391|HIP 87280|G...,Be*,z,Her,88.0,V744,267.513894,48.394152,3.1524,317.2190,pc,B6IIInp_sh,-0.100,<NA>,317.2190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.834259,48.394152,291.5452,Bpsh,-0.109,267.513885,3.430000,267.513894,48.394152,291.5452,3.1524,-0.1,NaN,B6IIInp_sh
4302,merged_hip:36778,36778,60606,<NA>,5587933566581112192,5587933566581112192,TIC 174008063|WISEA J073351.05-362018.0|HIP 36...,Be*,z,Pup,NaN,OW,113.462673,-36.338392,2.5769,388.0630,pc,B2Vne,-0.070,18000,388.0630,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.564179,-36.338392,363.6364,B2Vne,-0.078,113.462685,2.750000,113.462673,-36.338392,363.6364,2.5769,-0.07,18000.0,B2Vne
4303,merged_hip:45344,45344,<NA>,<NA>,5427085976191999616,5427085976191999616,TIC 75143198|GEN# +1.00079735|1RXS J091424.0-4...,*,z,Vel,NaN,None,138.602024,-43.227489,

In [452]:
# Broad stellar classification + final single-star table.
# Defaults for radius/color live in renderer (see notebooks/broad_stellar_classes.md).
def classify_from_otype(otype, spect=None):
    if pd.isna(otype):
        return None
    o = str(otype).strip()
    if not o:
        return None

    # Compact/remnant/substellar first.
    if re.match(r"^(Psr|N\*|NS|Magnetar|X)$", o):
        return "neutron_star_pulsar"
    if re.match(r"^(WD\*?|ELMWD)$", o):
        return "white_dwarf"
    if re.match(r"^(BD\*?|Pl\??)$", o):
        return "brown_dwarf_substellar"

    # Massive luminous blue.
    if re.match(r"^(WR\*?|WN\*?|WC\*?|WO\*?|LBV|s\*b)$", o):
        return "hot_massive_blue_luminous"

    # Blue MS and related pulsators/stragglers.
    if re.match(r"^(Be\*|bC\*|BS\*)$", o):
        return "blue_main_sequence"

    # Chemically peculiar / magnetic.
    if re.match(r"^(a2\*|Ap\*|Am\*|Bp\*|roAp|HgMn|He-strong|He-weak|SX\*)$", o):
        return "peculiar_chemically_peculiar"

    # Evolved cool/red.
    if re.match(r"^(RG\*|AGB\*|AB\*|Mi\*|LP\*|OH\*)$", o):
        return "red_giant_agb"
    if re.match(r"^(s\*r)$", o):
        return "red_supergiant_hypergiant"
    if re.match(r"^(C\*|cC\*|S\*|Ba\*)$", o):
        return "carbon_star"

    # Binaries: approximate primary from spectral type when available; else unresolved binary bucket.
    if re.match(r"^(\*\*\??|El\*|EB\*|SB\*|RS\*|Sy\*|XB\*|HMXB|LMXB|LX[B?]|HX[B?]|CV\*)$", o):
        spec_cat = classify_from_spect(spect)
        return spec_cat if spec_cat != "unknown" else "binary_multiple_unresolved"

    # Young stars.
    if re.match(r"^(Y\*O?|Or\*|TT\*|Ae\*|HH|out|FU\*)$", o):
        return "young_stellar_object"

    # Variability classes with known broad loci.
    if re.match(r"^(dS\*)$", o):
        return "yellow_white_main_sequence"
    if re.match(r"^(gD\*)$", o):
        return "yellow_white_main_sequence"
    if re.match(r"^(BY\*)$", o):
        spec_cat = classify_from_spect(spect)
        return spec_cat if spec_cat in {"orange_main_sequence", "red_dwarf"} else "unknown"
    if re.match(r"^(RR\*|WV\*)$", o):
        return "yellow_giant_bright_giant"
    if re.match(r"^(Ce\*)$", o):
        return "yellow_supergiant_hypergiant"

    # Generic star / PM / HV / Em / V: use spectral type fallback.
    if re.match(r"^(\*|PM\*|HV\*|Em\*|V\*|Ir\*|Er\*|Ro\*|Pu\*)$", o):
        return None

    return None


def classify_from_spect(spect):
    if pd.isna(spect):
        return "unknown"
    s = str(spect).strip().upper()
    if not s:
        return "unknown"

    if re.match(r"^D[A-Z]", s):
        return "white_dwarf"
    if re.match(r"^(C|S|CH|CN|CR|CJ)", s):
        return "carbon_star"
    if re.search(r"WR|WN|WC|WO", s):
        return "hot_massive_blue_luminous"

    m = re.search(r"[OBAFGKMLTY]", s)
    if not m:
        return "unknown"
    letter = m.group(0)

    if letter in {"L", "T", "Y"}:
        return "brown_dwarf_substellar"

    if re.search(r"IA\+|IA|IAB|IB|\bI\b", s):
        if letter in {"K", "M"}:
            return "red_supergiant_hypergiant"
        if letter in {"F", "G"}:
            return "yellow_supergiant_hypergiant"
        return "hot_massive_blue_luminous"

    if re.search(r"IV", s):
        if letter in {"O", "B"}:
            return "blue_subgiant"
        if letter == "A":
            return "white_subgiant"
        if letter in {"F", "G"}:
            return "yellow_subgiant"
        if letter in {"K", "M"}:
            return "orange_red_subgiant"
        return "yellow_subgiant"

    if re.search(r"II|III", s):
        if letter in {"F", "G", "K"}:
            return "yellow_giant_bright_giant"
        if letter == "M":
            return "red_giant_agb"
        return "hot_massive_blue_luminous" if letter in {"O", "B"} else "yellow_giant_bright_giant"

    # Main-sequence or no luminosity class fallback by temperature letter.
    if letter == "O":
        return "hot_massive_blue_luminous"
    if letter == "B":
        return "blue_main_sequence"
    if letter == "A":
        return "white_main_sequence"
    if letter == "F":
        return "yellow_white_main_sequence"
    if letter == "G":
        return "yellow_main_sequence"
    if letter == "K":
        return "orange_main_sequence"
    if letter == "M":
        return "red_dwarf"

    return "unknown"


def classify_star(row):
    return classify_from_otype(row.get("otype"), row.get("best_spect")) or classify_from_spect(row.get("best_spect"))


# Attach best source positions back to SIMBAD rows.
position_cols = ["source_key", "best_ra", "best_dec", "best_dist", "best_plx", "best_spect"]
simbad_work = simbad.copy()
simbad_work["source_key"] = simbad_work.apply(merged_source_key, axis=1)
simbad_work = simbad_work.merge(star_source_data[position_cols], on="source_key", how="left")

simbad_work["ra"] = simbad_work["best_ra"]
simbad_work["dec"] = simbad_work["best_dec"]
simbad_work["dist"] = simbad_work["best_dist"]
simbad_work["plx"] = simbad_work["best_plx"]
simbad_work["spect"] = simbad_work["best_spect"]


def fill_component_positions(group):
    group = group.copy()
    parent = group[group["is_main"].eq(True)]
    primary = group[group["is_primary"].eq(True)]

    # Reference: primary with most position values, else parent/main, else any row.
    candidate_indices = list(primary.index) + list(parent.index) + list(group.index)
    candidate_indices = list(dict.fromkeys(candidate_indices))
    candidates = group.loc[[idx for idx in candidate_indices if idx in group.index]].copy()
    candidates = candidates.assign(_pos_count=candidates[["ra", "dec", "dist"]].notna().sum(axis=1))
    ref = candidates.sort_values(["_pos_count"], ascending=False).iloc[0] if not candidates.empty else None

    if ref is not None:
        for idx, row in group.iterrows():
            # Primary may inherit from parent/system. Components may inherit missing pieces from primary/parent.
            for col in ["ra", "dec", "dist", "plx"]:
                if pd.isna(group.at[idx, col]) and pd.notna(ref.get(col)):
                    group.at[idx, col] = ref[col]

    # Avoid exact overlap within same Bayer group. If same RA/Dec, separate by tiny distance delta.
    # If same full position remains, add tiny deterministic angular offset too.
    seen_radec = {}
    seen_full = {}
    for idx in group.index:
        ra, dec, dist = group.at[idx, "ra"], group.at[idx, "dec"], group.at[idx, "dist"]
        if pd.notna(ra) and pd.notna(dec):
            key = (round(float(ra), 8), round(float(dec), 8))
            n = seen_radec.get(key, 0)
            if n and pd.notna(dist):
                group.at[idx, "dist"] = float(dist) * (1.0 + n * 1e-5)
            seen_radec[key] = n + 1

        ra, dec, dist = group.at[idx, "ra"], group.at[idx, "dec"], group.at[idx, "dist"]
        if pd.notna(ra) and pd.notna(dec) and pd.notna(dist):
            key = (round(float(ra), 8), round(float(dec), 8), round(float(dist), 8))
            n = seen_full.get(key, 0)
            if n:
                group.at[idx, "ra"] = (float(ra) + n * 0.00005) % 360
                group.at[idx, "dec"] = max(-90, min(90, float(dec) + n * 0.00003))
            seen_full[key] = n + 1

    return group.drop(columns=["_pos_count"], errors="ignore")


with_designation_parts = []
for _, group in simbad_work[simbad_work["designation_key"].notna()].groupby("designation_key", sort=False):
    with_designation_parts.append(fill_component_positions(group))
with_designation = pd.concat(with_designation_parts) if with_designation_parts else simbad_work.iloc[0:0].copy()
without_designation = simbad_work[simbad_work["designation_key"].isna()].copy()
simbad_positioned = pd.concat([with_designation, without_designation]).sort_index()

simbad_positioned["category"] = simbad_positioned.apply(classify_star, axis=1)

# Remove parent/system rows only when primary row exists and can take over.
# Keep main system row when only secondary/non-primary components exist (e.g. eta Sgr-like cases).
child_keys = set(simbad_positioned.loc[
    simbad_positioned["designation_key"].notna() & simbad_positioned["is_primary"].eq(True),
    "designation_key",
])
simbad_positioned["is_parent_system_only"] = (
    simbad_positioned["is_main"].eq(True)
    & ~simbad_positioned["is_primary"].eq(True)
    & simbad_positioned["designation_key"].isin(child_keys)
)

single_stars = simbad_positioned[~simbad_positioned["is_parent_system_only"]].copy()
single_stars["bf"] = np.where(single_stars["by"].notna() & single_stars["con"].notna(), single_stars["by"].astype(str) + " " + single_stars["con"].astype(str), None)
single_stars["flamsteed"] = np.where(single_stars["fl"].notna() & single_stars["fl_con"].notna(), single_stars["fl"].astype("Int64").astype(str) + " " + single_stars["fl_con"].astype(str), None)
single_stars["variable"] = np.where(single_stars["v"].notna() & single_stars["v_con"].notna(), single_stars["v"].astype(str) + " " + single_stars["v_con"].astype(str), None)

single_star_columns = [
    "hip_int", "merged_hip", "merged_hd", "merged_gl", "merged_gaia_dr3", "merged_gaia_dr2",
    "name_bad", "designation_key", "designation_kind", "bf", "by", "by_root", "by_ss", "by_component", "flamsteed", "fl", "fl_con", "fl_component", "variable", "v", "v_con", "v_component", "con",
    "ra", "dec", "dist", "plx", "spect", "otype", "category",
    "ids",
]
final_single_stars = single_stars[[c for c in single_star_columns if c in single_stars.columns]].copy()

missing_position_stars = final_single_stars[final_single_stars[["ra", "dec", "dist"]].isna().any(axis=1)].copy()
print("missing position rows:", len(missing_position_stars))

missing_position_stars
# final_single_stars





missing position rows: 11


,hip_int,merged_hip,merged_hd,merged_gl,merged_gaia_dr3,merged_gaia_dr2,name_bad,designation_key,designation_kind,bf,by,by_root,by_ss,by_component,flamsteed,fl,fl_con,fl_component,variable,v,v_con,v_component,con,ra,dec,dist,plx,spect,otype,category,ids
1267,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,47 Tuc,flamsteed,None,None,None,NaN,None,47 Tuc,47.0,Tuc,J,None,None,None,None,Tuc,NaN,NaN,NaN,NaN,,*,unknown,* 47 Tuc J
2390,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,chi CMa,bayer,chi CMa,chi,chi,NaN,None,None,NaN,None,None,None,None,None,None,CMa,NaN,NaN,NaN,NaN,,*,unknown,* chi CMa
2405,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,chi PsA,bayer,chi PsA,chi,chi,NaN,None,None,NaN,None,None,None,None,None,None,PsA,NaN,NaN,NaN,NaN,,*,unknown,* chi PsA
2739,<NA>,<NA>,93308,<NA>,5350358584482202880,5350358580171706624,,eta Car,bayer,eta Car,eta,eta,NaN,None,None,NaN,None,None,None,None,None,None,Car,161.264774,-59.684431,NaN,NaN,LBV,s*b,hot_massive_blue_luminous,TIC 458859916|LLNS 2725|Gaia DR2 535035858017...
3499,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,mu. Dor,bayer,mu. Dor,mu.,mu.,NaN,None,None,NaN,None,None,None,None,None,None,Dor,NaN,NaN,NaN,NaN,,*,unknown,* mu. Dor
3590,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,nu.02 Vel,bayer,nu.02 Vel,nu.02,nu.,2.0,None,None,NaN,None,None,None,None,None,None,Vel,NaN,NaN,NaN,NaN,,*,unknown,* nu.02 Vel
3609,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,nu. Crv,bayer,nu. Crv,nu.,nu.,NaN,None,None,NaN,None,None,None,None,None,None,Crv,NaN,NaN,NaN,NaN,,*,unknown,* nu. Crv
4064,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,sig02 Eri,bayer,sig02 Eri,sig02,sig,2.0,C,None,NaN,None,None,None,None,None,None,Eri,NaN,NaN,NaN,NaN,,*,unknown,* sig02 Eri C
4318,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,T Mon,bayer,T Mon,T,T,NaN,B,None,NaN,None,None,None,None,None,None,Mon,NaN,NaN,NaN,NaN,,*,unknown,* T Mon B
4319,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,T Mon,bayer,T Mon,T,T,NaN,C,None,NaN,None,None,None,None,None,None,Mon,NaN,NaN,NaN,NaN,,*,unknown,* T Mon C


In [453]:
# Manual fixes for missing/invalid position rows.
# Use final_single_stars row index from missing_position_stars output.
# Fill only values you trust. Leave dicts/lists empty to make no changes.

# Example:
# MANUAL_POSITION_FIXES = {
#     2390: {"ra": 123.456, "dec": -12.345, "dist": 100.0, "plx": 10.0},
# }
MANUAL_POSITION_FIXES = {
    # row_index: {"ra": None, "dec": None, "dist": None, "plx": None},
    2739: {"dist": 2300}, # eta Car
    
}

# Non-existent / invalid SIMBAD placeholder stars to remove from final table.
# Examples seen: chi CMa, chi PsA. Add row indexes from missing_position_stars here.
MANUAL_DROP_INDEXES = [
    1267,  # 47 Tuc
    2390,  # chi CMa
    2405,  # chi PsA
    3499,  # nu. Dor
    3590,  # nu.02 Vel
    3609,
    4064,
    4318,
    4319,
    4386,
]

# Optional: drop by exact common_name too, safer after common_name cell runs.
MANUAL_DROP_COMMON_NAMES = [
    # "Chi Canis Majoris",
    # "Chi Piscis Austrini",
]


def apply_manual_position_fixes(df, fixes=None, drop_indexes=None, drop_common_names=None):
    df = df.copy()
    fixes = fixes or {}
    drop_indexes = set(drop_indexes or [])
    drop_common_names = set(drop_common_names or [])

    for idx, values in fixes.items():
        if idx not in df.index:
            print(f"manual fix skipped, index not found: {idx}")
            continue
        for col in ["ra", "dec", "dist", "plx"]:
            if col in values and values[col] is not None:
                df.at[idx, col] = values[col]

    if drop_indexes:
        missing = sorted(drop_indexes - set(df.index))
        if missing:
            print(f"manual drops skipped, indexes not found: {missing}")
        df = df.drop(index=[idx for idx in drop_indexes if idx in df.index])

    if drop_common_names and "common_name" in df.columns:
        df = df[~df["common_name"].isin(drop_common_names)].copy()

    return df


final_single_stars = apply_manual_position_fixes(
    final_single_stars,
    fixes=MANUAL_POSITION_FIXES,
    drop_indexes=MANUAL_DROP_INDEXES,
    drop_common_names=MANUAL_DROP_COMMON_NAMES,
)

missing_position_stars = final_single_stars[final_single_stars[["ra", "dec", "dist"]].isna().any(axis=1)].copy()
print("missing position rows after manual fixes:", len(missing_position_stars))
missing_position_stars[[
    "merged_hip", "name_bad", "bf", "flamsteed", "variable", "con", "ra", "dec", "dist", "plx", "spect", "otype", "category", "ids"
]]



missing position rows after manual fixes: 0


,merged_hip,name_bad,bf,flamsteed,variable,con,ra,dec,dist,plx,spect,otype,category,ids


In [454]:
# Proper names + expanded common names.
# proper_name comes from IAU names by HIP. common_name prefers Bayer, then V*, then Flamsteed.

def constellation_genitive_map():
    return constellations.set_index("id")["gen"].to_dict()

CON_GENITIVE = constellation_genitive_map()


def first_notna(*values):
    for value in values:
        if pd.notna(value):
            return value
    return None


def normalize_hip_series(series):
    return pd.Series([numeric_id(x) for x in series], index=series.index, dtype="Int64")


def build_proper_name_map(iau_df):
    hip_col = next((c for c in iau_df.columns if c.lower() == "hip"), None)
    name_col = next((c for c in iau_df.columns if c.lower() in {"proper names", "proper name", "proper"}), None)
    if hip_col is None or name_col is None:
        return {}

    names = iau_df[[hip_col, name_col]].copy()
    names["hip_int"] = normalize_hip_series(names[hip_col])
    names = names[names["hip_int"].notna() & names[name_col].notna()]
    return names.drop_duplicates("hip_int").set_index("hip_int")[name_col].to_dict()


PROPER_NAME_BY_HIP = build_proper_name_map(iau_named)


def expanded_bayer_name(row):
    if pd.isna(row.get("by_root")) or pd.isna(row.get("con")):
        return None

    greek = SIMBAD_GREEK_ABBR.get(str(row["by_root"]), str(row["by_root"]))
    gen = CON_GENITIVE.get(row["con"], row["con"])

    if pd.notna(row.get("by_ss")):
        name = f"{greek} {int(row['by_ss'])} {gen}"
    else:
        name = f"{greek} {gen}"

    if pd.notna(row.get("by_component")):
        name = f"{name} {row['by_component']}"

    return name


def expanded_variable_name(row):
    if pd.isna(row.get("v")) or pd.isna(row.get("v_con")):
        return None

    gen = CON_GENITIVE.get(row["v_con"], row["v_con"])
    name = f"{row['v']} {gen}"

    if pd.notna(row.get("v_component")):
        name = f"{name} {row['v_component']}"

    return name


def expanded_flamsteed_name(row):
    if pd.isna(row.get("fl")) or pd.isna(row.get("fl_con")):
        return None

    gen = CON_GENITIVE.get(row["fl_con"], row["fl_con"])
    name = f"{int(row['fl'])} {gen}"

    if pd.notna(row.get("fl_component")):
        name = f"{name} {row['fl_component']}"

    return name


final_single_stars["proper_name"] = final_single_stars["merged_hip"].map(PROPER_NAME_BY_HIP)
final_single_stars["proper_name"] = final_single_stars["proper_name"].where(final_single_stars["proper_name"].notna(), None)

final_single_stars["bayer_name"] = final_single_stars.apply(expanded_bayer_name, axis=1)
final_single_stars["variable_name"] = final_single_stars.apply(expanded_variable_name, axis=1)
final_single_stars["flamsteed_name"] = final_single_stars.apply(expanded_flamsteed_name, axis=1)
final_single_stars["catalog_name"] = final_single_stars["merged_hip"].apply(
    lambda hip: f"HIP {int(hip)}" if pd.notna(hip) else None
)

final_single_stars["common_name"] = final_single_stars.apply(
    lambda row: first_notna(
        row["bayer_name"],
        row["variable_name"],
        row["flamsteed_name"],
        row["proper_name"],
        row["catalog_name"],
    ),
    axis=1,
)



def normalize_star_id(name):
    if pd.isna(name):
        return None
    text = strip_accents(str(name)).lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or None


CATEGORY_ENUM = [
    "unknown",
    "white_dwarf",
    "brown_dwarf_substellar",
    "red_dwarf",
    "orange_main_sequence",
    "yellow_main_sequence",
    "yellow_white_main_sequence",
    "white_main_sequence",
    "blue_main_sequence",
    "blue_subgiant",
    "white_subgiant",
    "yellow_subgiant",
    "orange_red_subgiant",
    "yellow_giant_bright_giant",
    "red_giant_agb",
    "red_supergiant_hypergiant",
    "yellow_supergiant_hypergiant",
    "hot_massive_blue_luminous",
    "carbon_star",
    "binary_multiple_unresolved",
    "peculiar_chemically_peculiar",
    "young_stellar_object",
    "neutron_star_pulsar",
]
CATEGORY_TO_ID = {category: i for i, category in enumerate(CATEGORY_ENUM)}

final_single_stars["id"] = final_single_stars["common_name"].apply(normalize_star_id)
final_single_stars["category_id"] = final_single_stars["category"].map(CATEGORY_TO_ID).fillna(0).astype(int)

# Put name columns up front; keep no renderer-default radius/color columns here.
front_cols = [
    "id", "hip_int", "merged_hip", "proper_name", "common_name", "bayer_name", "variable_name", "flamsteed_name", "catalog_name",
    "merged_hd", "merged_gl", "merged_gaia_dr3", "merged_gaia_dr2", "category", "category_id",
]
remaining_cols = [c for c in final_single_stars.columns if c not in front_cols]
final_single_stars = final_single_stars[front_cols + remaining_cols]

missing_common_name_stars = final_single_stars[final_single_stars["common_name"].isna()].copy()
missing_designation_name_stars = final_single_stars[
    final_single_stars[["bayer_name", "variable_name", "flamsteed_name", "proper_name"]].isna().all(axis=1)
].copy()
print("missing common_name rows:", len(missing_common_name_stars))
print("common_name used catalog fallback rows:", len(missing_designation_name_stars))

final_single_stars




missing common_name rows: 0
common_name used catalog fallback rows: 2


,id,hip_int,merged_hip,proper_name,common_name,bayer_name,variable_name,flamsteed_name,catalog_name,merged_hd,merged_gl,merged_gaia_dr3,merged_gaia_dr2,category,category_id,name_bad,designation_key,designation_kind,bf,by,by_root,by_ss,by_component,flamsteed,fl,fl_con,fl_component,variable,v,v_con,v_component,con,ra,dec,dist,plx,spect,otype,ids
0,100_aquarii,116118,116118,None,100 Aquarii,None,None,100 Aquarii,HIP 116118,221357,<NA>,2388538245106801280,2388538245106801280,yellow_white_main_sequence,6,,100 Aqr,flamsteed,None,None,None,NaN,None,100 Aqr,100.0,Aqr,None,None,None,None,None,Aqr,352.925158,-21.369462,76.1615,13.6139,F0V,*,Gaia DR3 2388538245106801280|* 100 Aqr|BD-22 ...
2,100_herculis_a,88818,88818,None,100 Herculis A,None,None,100 Herculis A,HIP 88818,166045,<NA>,4579990675911466624,4579990675911466624,binary_multiple_unresolved,19,,100 Her,flamsteed,None,None,None,NaN,None,100 Her,100.0,Her,A,None,None,None,None,Her,271.956667,26.101111,49.603696,20.15999,,**,TIC 320932914|HIP 88818|Gaia DR3 4579990675911...
3,100_herculis_b,88817,88817,None,100 Herculis B,None,None,100 Herculis B,HIP 88817,166046,<NA>,4579990675911464704,4579990675911464704,white_main_sequence,7,,100 Her,flamsteed,None,None,None,NaN,None,100 Her,100.0,Her,B,None,None,None,None,Her,271.956261,26.097333,38.5802,15.6586,A3V,*,TIC 320932916|HIP 88817|Gaia DR3 4579990675911...
4,100_piscium,7364,7364,None,100 Piscium,None,None,100 Piscium,HIP 7364,9656,<NA>,2586131788972205696,2586131788972205696,white_main_sequence,7,,100 Psc,flamsteed,None,None,None,NaN,None,100 Psc,100.0,Psc,None,None,None,None,None,Psc,23.715089,12.558636,231.4815,10.3996,A3V,*,TIC 47119710|IDS 01295+1203 A|HIP 7364|Gaia DR...
5,101_herculis,88899,88899,None,101 Herculis,None,None,101 Herculis,HIP 88899,166230,<NA>,4527896021146097408,4527896021146097408,yellow_giant_bright_giant,13,,101 Her,flamsteed,None,None,None,NaN,None,101 Her,101.0,Her,None,None,None,None,None,Her,272.220266,20.045233,100.8065,9.5493,A8III,*,TIC 296081957|HIP 88899|Gaia DR3 4527896021146...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4534,zeta_volantis_b,<NA>,<NA>,None,Zeta Volantis B,Zeta Volantis B,None,None,None,<NA>,<NA>,5263150888430032384,5263150888430032384,unknown,0,,zet Vol,bayer,zet Vol,zet,zet,NaN,B,None,NaN,None,None,None,None,None,None,Vol,115.469059,-72.608255,44.222,22.613,,*,Gaia DR2 5263150888430032384|TIC 272086940|Gai...
4535,z_herculis,87280,87280,None,z Herculis,z Herculis,V744 Herculis,88 Herculis,HIP 87280,162732,<NA>,1363284299777747584,1363284299777747584,blue_main_sequence,8,,z Her,bayer,z Her,z,z,NaN,None,88 Her,88.0,Her,None,V744 Her,V744,Her,None,Her,267.513894,48.394152,291.5452,3.1524,B6IIInp_sh,Be*,TIC 356239023|AP J17500331+4823391|HIP 87280|G...
4536,z_puppis,36778,36778,None,z Puppis,z Puppis,OW Puppis,None,HIP 36778,60606,<NA>,5587933566581112192,5587933566581112192,blue_main_sequence,8,,z Pup,bayer,z Pup,z,z,NaN,None,None,NaN,None,None,OW Pup,OW,Pup,None,Pup,113.462673,-36.338392,363.6364,2.5769,B2Vne,Be*,TIC 174008063|WISEA J073351.05-362018.0|HIP 36...
4537,z_velorum_a,45344,45344,None,z Velorum A,z Velorum A,None,None,HIP 45344,<NA>,<NA>,5427085976191999616,5427085976191999616,blue_main_sequence,8,,z Vel,bayer,z Vel,z,z,NaN,A,None,NaN,None,None,None,None,None,None,Vel,138.602024,-43.227489,187.6173,5.3395,B3/5V,*,TIC 75143198|GEN# +1.00079735|1RXS J091424.0-4...


In [455]:
missing_designation_name_stars

,id,hip_int,merged_hip,proper_name,common_name,bayer_name,variable_name,flamsteed_name,catalog_name,merged_hd,merged_gl,merged_gaia_dr3,merged_gaia_dr2,category,category_id,name_bad,designation_key,designation_kind,bf,by,by_root,by_ss,by_component,flamsteed,fl,fl_con,fl_component,variable,v,v_con,v_component,con,ra,dec,dist,plx,spect,otype,ids
3014,hip_109754,109754,109754,None,HIP 109754,None,None,None,HIP 109754,211073,<NA>,1955662200884781824,1955662200880678272,yellow_giant_bright_giant,13,,None,None,None,None,None,NaN,None,None,NaN,None,None,None,None,None,None,None,333.469698,39.714925,147.929,5.7181,K2.5III,**,TIC 119763135|Gaia DR3 1955662200884781824|PLX...
3018,hip_44700,44700,44700,None,HIP 44700,None,None,None,HIP 44700,77912,<NA>,719103698606421888,719103698606421888,yellow_supergiant_hypergiant,16,,None,None,None,None,None,NaN,None,None,NaN,None,None,None,None,None,None,None,136.632366,38.452214,250.6266,4.6253,G7IIa,Pe*,TIC 16148554|HIP 44700|Gaia DR3 71910369860642...


In [456]:
# Check common_name uniqueness. common_name will be used as star id/index.
common_name_counts = final_single_stars["common_name"].value_counts(dropna=False)
duplicate_common_names = final_single_stars[
    final_single_stars["common_name"].isin(common_name_counts[common_name_counts > 1].index)
].sort_values(["common_name", "merged_hip", "hip_int"])

print("duplicate common_name rows:", len(duplicate_common_names))
print("duplicate common_name values:", int((common_name_counts > 1).sum()))

duplicate_common_names[[
    "common_name", "proper_name", "merged_hip", "hip_int", "designation_key", "designation_kind", "ra", "dec", "dist", "category", "ids"
]]



duplicate common_name rows: 0
duplicate common_name values: 0


,common_name,proper_name,merged_hip,hip_int,designation_key,designation_kind,ra,dec,dist,category,ids


In [457]:
# Map constellation path HIP ids to final star ids.
# Keep original line/path structure, but replace each HIP with normalized final_single_stars.id via merged_hip.
star_id_by_merged_hip = (
    final_single_stars[final_single_stars["merged_hip"].notna()]
    .drop_duplicates("merged_hip")
    .set_index("merged_hip")["id"]
    .to_dict()
)
star_id_by_hip = (
    final_single_stars[final_single_stars["hip_int"].notna()]
    .drop_duplicates("hip_int")
    .set_index("hip_int")["id"]
    .to_dict()
)

hip_to_star_id = {}
for _, row in constellation_hip_matches.iterrows():
    hip = int(row["hip_int"])
    merged_hip = row.get("merged_hip")
    if pd.notna(merged_hip) and int(merged_hip) in star_id_by_merged_hip:
        hip_to_star_id[hip] = star_id_by_merged_hip[int(merged_hip)]
    elif hip in star_id_by_hip:
        hip_to_star_id[hip] = star_id_by_hip[hip]


def map_hip_paths_to_star_ids(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return value
    if isinstance(value, list):
        return [map_hip_paths_to_star_ids(x) for x in value]
    if isinstance(value, tuple):
        return tuple(map_hip_paths_to_star_ids(x) for x in value)
    try:
        hip = int(value)
    except (TypeError, ValueError):
        return value
    return hip_to_star_id.get(hip, f"hip_{hip}")

# Notebook table uses lines_hip; create requested paths_id plus alias lines_id.
constellations["paths_id"] = constellations["lines_hip"].apply(map_hip_paths_to_star_ids)
constellations["lines_id"] = constellations["paths_id"]
constellations["bright"] = constellations["brightest_hip"].apply(
    lambda hip: hip_to_star_id.get(int(hip), f"hip_{int(hip)}") if pd.notna(hip) else None
)

missing_path_name_hips = sorted(set(constellation_path_hips) - set(hip_to_star_id))
print("constellation HIPs without star id mapping:", len(missing_path_name_hips))
missing_path_name_hips[:50], constellations[["id", "lines_hip", "paths_id", "brightest_hip", "bright"]]



constellation HIPs without star id mapping: 0


/tmp/ipykernel_1782992/3177220802.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  constellations["paths_id"] = constellations["lines_hip"].apply(map_hip_paths_to_star_ids)
/tmp/ipykernel_1782992/3177220802.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  constellations["lines_id"] = constellations["paths_id"]
/tmp/ipykernel_1782992/3177220802.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

Se

([],
      id                                          lines_hip  \
 0   And  [[677, 3092, 5447, 9640], [113726, 116631, 116...   
 1   Ant                            [[53502, 51172, 46515]]   
 2   Aps            [[72370, 81065], [80047, 81852, 81065]]   
 3   Aql  [[98036, 97649, 97278, 95501, 93805, 95501, 93...   
 4   Aqr  [[102618, 106278, 109074, 110395, 110960, 1114...   
 5   Ara  [[85267, 85727, 82363, 83081, 83153, 85792, 88...   
 6   Ari                        [[8832, 8903, 9884, 13209]]   
 7   Aur  [[25428, 23015, 23767, 24608, 28360, 28380, 25...   
 8   Boo  [[69673, 72105, 74666, 73555, 71075, 71053, 69...   
 9   CMa  [[30324, 32349, 34444, 33579, 34444, 35904], [...   
 10  CMi                                   [[37279, 36188]]   
 11  CVn                                   [[63125, 61317]]   
 12  Cae                     [[23595, 21861, 21770, 21060]]   
 13  Cam  [[23040, 23522, 22783, 29997, 33694, 29997, 22...   
 14  Cap  [[100064, 100345, 102485, 102978, 105881

In [458]:
# Write constellation JSON files.
# Output: public/data/constellations/<constellation id lowercase>.json

def json_clean(value):
    if pd.isna(value) if not isinstance(value, (list, tuple, dict, np.ndarray)) else False:
        return None
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return [json_clean(x) for x in value.tolist()]
    if isinstance(value, (list, tuple)):
        return [json_clean(x) for x in value]
    if isinstance(value, dict):
        return {str(k): json_clean(v) for k, v in value.items()}
    return value

# Resolve repo root whether notebook kernel cwd is repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

constellation_out_dir = PROJECT_ROOT / "public/data/constellations"
constellation_out_dir.mkdir(parents=True, exist_ok=True)

constellation_json_rows = []
for _, row in constellations.sort_values("id").iterrows():
    item = {
        "id": row["id"],
        "name": row["name"],
        "gen": row["gen"],
        "meaning": row["meaning"],
        "rank": int(row["rank"]),
        "focus": json_clean(row["focus"]),
        "label": json_clean(row["label"]),
        "bound": json_clean(row["boundaries"]),
        "lines": json_clean(row["paths_id"]),
        "bright": json_clean(row["bright"]),
    }
    constellation_json_rows.append(item)
    # (constellation_out_dir / f"{row['id'].lower()}.json").write_text(json.dumps(item, ensure_ascii=False, separators=(",", ":"))) # compact
    (constellation_out_dir / f"{row['id'].lower()}.json").write_text(json.dumps(item, ensure_ascii=False, indent=2)) # nice format

# Requested schema path. This schema describes exported constellation JSON files.
constellation_schema = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "title": "Constellation",
    "type": "object",
    "required": ["id", "name", "gen", "meaning", "rank", "focus", "label", "bound", "lines", "bright"],
    "properties": {
        "id": {"type": "string"},
        "name": {"type": "string"},
        "gen": {"type": "string"},
        "meaning": {"type": "string"},
        "rank": {"type": "integer"},
        "focus": {"type": ["array", "null"], "items": {"type": "number"}},
        "label": {"type": ["array", "null"], "items": {"type": "number"}},
        "bound": {"type": ["array", "null"]},
        "lines": {"type": ["array", "null"]},
        "bright": {"type": ["string", "null"]},
    },
    "additionalProperties": False,
}
schema_path = PROJECT_ROOT / "public/data/constellations.schema.json"
schema_path.parent.mkdir(parents=True, exist_ok=True)
schema_path.write_text(json.dumps(constellation_schema, indent=2))

print("cwd:", Path.cwd())
print("project root:", PROJECT_ROOT)
print("wrote constellation files:", len(constellation_json_rows))
print("constellation dir:", constellation_out_dir)
print("schema:", schema_path)
print("sample file exists:", (constellation_out_dir / "and.json").exists())
# constellation_json_rows[:3]




cwd: /home/sebl/code/etoile/notebooks
project root: /home/sebl/code/etoile
wrote constellation files: 88
constellation dir: /home/sebl/code/etoile/public/data/constellations
schema: /home/sebl/code/etoile/public/data/constellations.schema.json
sample file exists: True


In [459]:
# Write star JSON files and star category defaults.
# Outputs:
# - public/data/stars/<star id>.json
# - public/data/stars.schema.json
# - public/data/star-categories/<category enum id>.json
# - public/data/star-categories.schema.json

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

stars_out_dir = PROJECT_ROOT / "public/data/stars"
stars_out_dir.mkdir(parents=True, exist_ok=True)
star_categories_out_dir = PROJECT_ROOT / "public/data/star-categories"
star_categories_out_dir.mkdir(parents=True, exist_ok=True)

SUPERSCRIPT_DIGITS = str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹")
GREEK_SYMBOLS = {
    "alf": "α", "bet": "β", "gam": "γ", "del": "δ", "eps": "ε", "zet": "ζ", "eta": "η",
    "the": "θ", "tet": "θ", "iot": "ι", "kap": "κ", "lam": "λ", "mu.": "μ", "mu": "μ",
    "nu.": "ν", "nu": "ν", "ksi": "ξ", "omi": "ο", "pi.": "π", "pi": "π", "rho": "ρ",
    "sig": "σ", "tau": "τ", "ups": "υ", "phi": "φ", "khi": "χ", "chi": "χ", "psi": "ψ", "ome": "ω",
}


def clean_optional(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    return value


def required_float(value):
    value = clean_optional(value)
    return None if value is None else float(value)


def optional_int(value):
    value = clean_optional(value)
    return None if value is None else int(value)


def by_to_greek(row):
    if pd.isna(row.get("by_root")):
        return None
    root = str(row["by_root"])
    symbol = GREEK_SYMBOLS.get(root, root)
    if pd.notna(row.get("by_ss")):
        n = str(int(row["by_ss"]))
        return symbol + n.translate(SUPERSCRIPT_DIGITS)
    return symbol


def omit_none(d):
    return {k: v for k, v in d.items() if v is not None}


# Star rank: proper=2, constellation-line=3, Bayer=4, otherwise=5.
line_star_ids = set()
for value in constellations.get("paths_id", []):
    for x in flatten_hips(value):
        # flatten_hips only returns ints, so use custom string walk below instead.
        pass

def flatten_strings(value):
    out = []
    def walk(x):
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return
        if isinstance(x, (list, tuple, set)):
            for item in x:
                walk(item)
            return
        out.append(str(x))
    walk(value)
    return out

for paths in constellations["paths_id"].dropna():
    line_star_ids.update(flatten_strings(paths))

final_single_stars["rank"] = 5
final_single_stars.loc[final_single_stars["by"].notna(), "rank"] = 4
final_single_stars.loc[final_single_stars["id"].isin(line_star_ids), "rank"] = 3
final_single_stars.loc[final_single_stars["proper_name"].notna(), "rank"] = 2
final_single_stars["rank"] = final_single_stars["rank"].astype(int)

# Write one star per file.
star_json_rows = []
for _, row in final_single_stars.sort_values("id").iterrows():
    item = {
        "id": str(row["id"]),
        "name": str(row["common_name"]),
        "rank": int(row["rank"]),
        "ra": required_float(row.get("ra")),
        "dec": required_float(row.get("dec")),
        "dist": required_float(row.get("dist")),
        "otype": clean_optional(row.get("otype")),  # required, null when missing
        "spect": clean_optional(row.get("spect")),  # required, null when missing
        "cat": int(row.get("category_id", 0)),
    }
    proper = clean_optional(row.get("proper_name"))
    if proper is not None:
        item["proper"] = proper

    if pd.notna(row.get("by")) and pd.notna(row.get("con")):
        item.update({
            "by": str(row["by"]),
            "by_greek": by_to_greek(row),
            "by_con": str(row["con"]),
        })
    if pd.notna(row.get("by_component")):
        item["by_comp"] = str(row["by_component"])

    if pd.notna(row.get("v")) and pd.notna(row.get("v_con")):
        item.update({"v": str(row["v"]), "v_con": str(row["v_con"])})
    if pd.notna(row.get("v_component")):
        item["v_comp"] = str(row["v_component"])

    if pd.notna(row.get("fl")) and pd.notna(row.get("fl_con")):
        item.update({"fl": int(row["fl"]), "fl_con": str(row["fl_con"])})
    if pd.notna(row.get("fl_component")):
        item["fl_comp"] = str(row["fl_component"])

    # Drop only optional nulls. Required nullable fields otype/spect stay present.
    optional_keys = ["proper", "by", "by_greek", "by_con", "by_comp", "v", "v_con", "v_comp", "fl", "fl_con", "fl_comp"]
    for key in optional_keys:
        if key in item and item[key] is None:
            del item[key]
    star_json_rows.append(item)
    (stars_out_dir / f"{item['id']}.json").write_text(json.dumps(item, ensure_ascii=False, indent=2))

star_schema = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "title": "Star",
    "type": "object",
    "required": ["id", "name", "rank", "ra", "dec", "dist", "otype", "spect", "cat"],
    "properties": {
        "id": {"type": "string"},
        "name": {"type": "string"},
        "proper": {"type": "string"},
        "rank": {"type": "integer", "minimum": 2, "maximum": 5},
        "by": {"type": "string"},
        "by_greek": {"type": "string"},
        "by_con": {"type": "string"},
        "by_comp": {"type": "string"},
        "v": {"type": "string"},
        "v_con": {"type": "string"},
        "v_comp": {"type": "string"},
        "fl": {"type": "integer"},
        "fl_con": {"type": "string"},
        "fl_comp": {"type": "string"},
        "ra": {"type": "number"},
        "dec": {"type": "number"},
        "dist": {"type": "number"},
        "otype": {"type": ["string", "null"]},
        "spect": {"type": ["string", "null"]},
        "cat": {"type": "integer", "minimum": 0},
    },
    "dependentRequired": {
        "by": ["by_greek", "by_con"],
        "by_greek": ["by", "by_con"],
        "by_con": ["by", "by_greek"],
        "v": ["v_con"],
        "v_con": ["v"],
        "fl": ["fl_con"],
        "fl_con": ["fl"],
    },
    "additionalProperties": False,
}
(PROJECT_ROOT / "public/data/stars.schema.json").write_text(json.dumps(star_schema, indent=2))

# Category defaults for GPU/Three.js shader lookup. id matches star.cat enum.
CATEGORY_RENDER_DEFAULTS = {
    "unknown": {"rgb": [230, 230, 230], "radius": 1.0, "point": 1.0, "halo": 1.15, "glow": 0.25},
    "white_dwarf": {"rgb": [210, 225, 255], "radius": 0.01, "point": 0.35, "halo": 0.7, "glow": 0.35},
    "brown_dwarf_substellar": {"rgb": [170, 80, 45], "radius": 0.1, "point": 0.5, "halo": 0.75, "glow": 0.15},
    "red_dwarf": {"rgb": [255, 120, 80], "radius": 0.3, "point": 0.7, "halo": 0.9, "glow": 0.2},
    "orange_main_sequence": {"rgb": [255, 170, 90], "radius": 0.7, "point": 0.9, "halo": 1.05, "glow": 0.25},
    "yellow_main_sequence": {"rgb": [255, 230, 140], "radius": 1.0, "point": 1.0, "halo": 1.15, "glow": 0.3},
    "yellow_white_main_sequence": {"rgb": [255, 245, 200], "radius": 1.3, "point": 1.1, "halo": 1.25, "glow": 0.32},
    "white_main_sequence": {"rgb": [245, 248, 255], "radius": 2.0, "point": 1.2, "halo": 1.35, "glow": 0.35},
    "blue_main_sequence": {"rgb": [170, 205, 255], "radius": 5.0, "point": 1.4, "halo": 1.6, "glow": 0.45},
    "blue_subgiant": {"rgb": [165, 200, 255], "radius": 8.0, "point": 1.55, "halo": 1.8, "glow": 0.5},
    "white_subgiant": {"rgb": [245, 248, 255], "radius": 3.0, "point": 1.3, "halo": 1.5, "glow": 0.38},
    "yellow_subgiant": {"rgb": [255, 235, 170], "radius": 2.5, "point": 1.25, "halo": 1.45, "glow": 0.36},
    "orange_red_subgiant": {"rgb": [255, 170, 100], "radius": 4.0, "point": 1.35, "halo": 1.55, "glow": 0.35},
    "yellow_giant_bright_giant": {"rgb": [255, 210, 130], "radius": 25.0, "point": 1.8, "halo": 2.1, "glow": 0.45},
    "red_giant_agb": {"rgb": [255, 150, 90], "radius": 100.0, "point": 2.1, "halo": 2.5, "glow": 0.5},
    "red_supergiant_hypergiant": {"rgb": [255, 105, 70], "radius": 700.0, "point": 2.7, "halo": 3.4, "glow": 0.65},
    "yellow_supergiant_hypergiant": {"rgb": [255, 220, 120], "radius": 300.0, "point": 2.45, "halo": 3.0, "glow": 0.6},
    "hot_massive_blue_luminous": {"rgb": [155, 195, 255], "radius": 80.0, "point": 2.2, "halo": 2.8, "glow": 0.75},
    "carbon_star": {"rgb": [255, 95, 55], "radius": 200.0, "point": 2.1, "halo": 2.7, "glow": 0.55},
    "binary_multiple_unresolved": {"rgb": [255, 255, 220], "radius": 1.5, "point": 1.15, "halo": 1.4, "glow": 0.32},
    "peculiar_chemically_peculiar": {"rgb": [220, 230, 255], "radius": 1.5, "point": 1.1, "halo": 1.35, "glow": 0.4},
    "young_stellar_object": {"rgb": [255, 190, 120], "radius": 2.0, "point": 1.2, "halo": 1.7, "glow": 0.5},
    "neutron_star_pulsar": {"rgb": [180, 220, 255], "radius": 0.00002, "point": 0.25, "halo": 0.6, "glow": 0.8},
}

category_json_rows = []
for name in CATEGORY_ENUM:
    cid = CATEGORY_TO_ID[name]
    d = CATEGORY_RENDER_DEFAULTS[name]
    item = {
        "id": cid,
        "name": name,
        "rgb": d["rgb"],
        "color": [round(x / 255, 6) for x in d["rgb"]],
        "radius": d["radius"],
        "point": d["point"],
        "halo": d["halo"],
        "glow": d["glow"],
    }
    category_json_rows.append(item)
    (star_categories_out_dir / f"{cid}.json").write_text(json.dumps(item, ensure_ascii=False, indent=2))

category_schema = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "title": "StarCategory",
    "type": "object",
    "required": ["id", "name", "rgb", "color", "radius", "point", "halo", "glow"],
    "properties": {
        "id": {"type": "integer", "minimum": 0},
        "name": {"type": "string"},
        "rgb": {"type": "array", "minItems": 3, "maxItems": 3, "items": {"type": "integer", "minimum": 0, "maximum": 255}},
        "color": {"type": "array", "minItems": 3, "maxItems": 3, "items": {"type": "number", "minimum": 0, "maximum": 1}},
        "radius": {"type": "number"},
        "point": {"type": "number"},
        "halo": {"type": "number"},
        "glow": {"type": "number"},
    },
    "additionalProperties": False,
}
(PROJECT_ROOT / "public/data/star-categories.schema.json").write_text(json.dumps(category_schema, indent=2))

print("wrote star files:", len(star_json_rows), "->", stars_out_dir)
print("wrote category files:", len(category_json_rows), "->", star_categories_out_dir)
print("star schema:", PROJECT_ROOT / "public/data/stars.schema.json")
print("category schema:", PROJECT_ROOT / "public/data/star-categories.schema.json")
# star_json_rows[:3], category_json_rows[:3]




wrote star files: 4307 -> /home/sebl/code/etoile/public/data/stars
wrote category files: 23 -> /home/sebl/code/etoile/public/data/star-categories
star schema: /home/sebl/code/etoile/public/data/stars.schema.json
category schema: /home/sebl/code/etoile/public/data/star-categories.schema.json


In [460]:
con_mismatch = final_single_stars[
   (
       final_single_stars["fl_con"].notna()
       & final_single_stars["con"].notna()
       & final_single_stars["fl_con"].ne(final_single_stars["con"])
   )
   | (
       final_single_stars["v_con"].notna()
       & final_single_stars["con"].notna()
       & final_single_stars["v_con"].ne(final_single_stars["con"])
   )
].copy()

con_mismatch[[
   "id",
   "common_name",
   "con",
   "flamsteed",
   "fl_con",
   "variable",
   "v_con",
   "bf",
   "designation_key",
   "designation_kind",
   "ids",
]].sort_values(["con", "fl_con", "v_con", "common_name"])

,id,common_name,con,flamsteed,fl_con,variable,v_con,bf,designation_key,designation_kind,ids
1594,v520_persei,V520 Persei,And,61 And,And,V520 Per,Per,None,61 And,flamsteed,TIC 264730431|HIP 10805|Gaia DR3 4583747883367...
4397,vy_piscium,VY Piscium,Ari,3 Ari,Ari,VY Psc,Psc,None,3 Ari,flamsteed,TIC 88773937|HIP 8271|Gaia DR3 916517201742549...
3934,psi_10_aurigae,Psi 10 Aurigae,Aur,16 Lyn,Lyn,None,None,psi10 Aur,psi10 Aur,bayer,TIC 192062418|Gaia DR3 953838195501646848|HIP ...
2980,g_canis_minoris,G Canis Minoris,CMi,13 Pup,Pup,None,None,G CMi,G CMi,bayer,TIC 452867458|AP J08021594+0220044|HIP 39311|G...
1430,ae_lyncis,AE Lyncis,Cam,54 Cam,Cam,AE Lyn,Lyn,None,54 Cam,flamsteed,TIC 80882380|HIP 39348|Gaia DR3 10815650940460...
1492,os_ursae_majoris,OS Ursae Majoris,Cam,57 Cam,Cam,OS UMa,UMa,None,57 Cam,flamsteed,TIC 73315277|HIP 40772|Gaia DR3 10903672000257...
1549,ap_piscium,AP Piscium,Cet,5 Cet,Cet,AP Psc,Psc,None,5 Cet,flamsteed,TIC 300966856|HIP 664|Gaia DR3 244844465511088...
2493,delta_columbae,Delta Columbae,Col,3 CMa,CMa,None,None,del Col,del Col,bayer,TIC 124939805|HIP 30277|Gaia DR3 2891816671700...
834,ty_corvi,TY Corvi,Crt,31 Crt,Crt,TY Crv,Crv,None,31 Crt,flamsteed,TIC 428684629|HIP 58587|Gaia DR3 3518794554458...
2123,alpha_fornacis,Alpha Fornacis,For,12 Eri,Eri,None,None,alf For,alf For,bayer,Gaia DR2 5059348952156075776|TIC 88523071|GC ...


In [461]:
QTable.from_pandas(final_single_stars[["hip_int", "merged_hip", "proper_name", "common_name", "con", "by", "by_root", "by_ss", "by_component", "fl", "fl_component", "v", "v_component", "otype", "category", "ra", "dec", "dist", "spect"]]).show_in_browser(jsviewer=True)
